# OI-биочип v4: нейроны Pospischil, HD-MEA, гель, R-STDP (дискриминация A/B) + Pong v3

Пакет `oipong` версии 4. Движок v3 (Pong: `OIPong`, `make_config`, `oi_engine`) сохранён без изменений
интерфейсов. Рядом с ним новый движок уровня 2: проводимостные нейроны RS/FS, сеть из ≥1000 клеток над
матрицей электродов, гелевая среда, обучение STDP + след пригодности + дофамин + синаптическое масштабирование,
информационные метрики и панель визуализации V1–V6.

Запуск в Colab: `OI_Pong_3D_Biochip_v4.ipynb`. Ячейка 1 записывает пакет, дальше по порядку.
Запуск на сервере: `python run_checks.py fi | ab | viz | env` (см. «Команды»).


Ячейки 1–6 — движок уровня 2 (v4). Ячейки после заголовка «Pong v3» — игра на движке v3 без изменений (их нумерация в комментариях — по ноутбуку v3).

In [ ]:
# Ячейка 1: записывает пакет oipong/ (v4), oi_v4_checks.py и oi_engine.py. Запускать один раз за сессию.
import os, sys
os.makedirs('oipong', exist_ok=True)
FILES = {"__init__.py": "\"\"\"oipong v4: 3D-симулятор органоидного интеллекта (биочип на HD-MEA).\n\nДвижок v3 (Pong, без изменений интерфейсов):\n  config, hh (Ходжкин–Хаксли 1952), environment (BioEnvironment: Q10, глутамат, ГАМК, дофамин), network, game,\n  learning (RM-Hebb), sim (OIPong), experiments.\nДвижок уровня 2 (новое в v4):\n  bioconfig (BioConfig), neuron (Pospischil et al. 2008: Na m^3h, Kd n^4, M-ток p, утечка; RS/FS),\n  gel (температура Q10, блок NMDA Mg2+ по Jahr & Stevens 1990, вязкость -> радиус связности, диффузия K+),\n  mea (HD-MEA: считывание с затуханием, гауссова стимуляция), bionet (3D-сеть, AMPA/NMDA/ГАМК-А),\n  plasticity (STDP + след пригодности + дофамин + синаптическое масштабирование), task_ab (дискриминация A/B),\n  info (MI, TE), viz (панель V1–V6, анимация; импортируется отдельно: from oipong import viz).\n\"\"\"\n__version__ = \"4.0\"\n\nfrom .config import (Config, PRESETS, make_config, cfg_to_dict, cfg_from_dict, save_config, load_config, log_run,\n                     ROLE_SENS, ROLE_HID, ROLE_MOT, ROLE_NAME)\nfrom .environment import BioEnvironment\nfrom .sim import OIPong, save_game, load_game\nfrom .experiments import (baseline_winrate, make_oracle, weight_structure, benchmark, ablation, debug_ladder,\n                          temperature_scan, dopamine_scan)\n# уровень 2\nfrom .bioconfig import BioConfig, make_bio_config, bio_cfg_to_dict, save_bio_config\nfrom .neuron import NeuronArray, RS, FS, TYPE_NAME, CELL_PARAMS, fi_curve, simulate_single\nfrom .gel import GelMedium, mg_block, q10_factor, conn_radius_from_viscosity\nfrom .mea import HDMEA\nfrom .bionet import BioNet, Recorder\nfrom .plasticity import RSTDP\nfrom .task_ab import ABTask, run_ab_experiment, dprime\nfrom .info import mutual_information, transfer_entropy, mi_with_surrogate, te_with_surrogate, sensory_motor_info\nimport numpy as np\nBOX = np.array(Config.box)   # для совместимости с ячейками визуализации v3 (размер матрицы в cfg.box)\n", "config.py": "\"\"\"Конфигурация, пресеты, сохранение/загрузка настроек и журнал запусков.\"\"\"\nimport json\nimport time\n\nROLE_SENS, ROLE_HID, ROLE_MOT = 0, 1, 2\nROLE_NAME = {0: \"sensory\", 1: \"hidden\", 2: \"motor\"}\n\n\nclass Config:\n    \"\"\"Все параметры в одном месте. Меняйте через make_config(preset, **kw).\"\"\"\n    seed = 7\n\n    # ---- геометрия и состав сети\n    box = (800.0, 800.0, 600.0)     # мкм, гидрогелевая матрица\n    n_sens = 8                       # сенсорные электроды\n    n_hid0 = 14                      # скрытые нейроны на старте\n    n_mot_each = 8                   # нейронов в группе UP и в группе DOWN\n    hid_min, hid_max = 8, 36\n    direct_p = 1.0                   # доля прямых связей сенсор -> мотор\n    min_deg = 4\n    w_init = (0.15, 0.5)\n    conn_radius = 230.0              # мкм\n    conn_lambda = 110.0\n    inhib_frac = 0.2\n\n    # ---- время\n    frame_ms = 10.0                  # один игровой кадр\n    dt = 0.05                        # шаг интегрирования HH, мс\n\n    # ---- игра\n    ball_vx = 1.0 / 12.0\n    paddle_gain = 0.015              # смещение ракетки за 1 спайк разницы UP-DOWN\n    paddle_max = 0.09\n    hit_half = 0.12\n    encoding = \"signed\"              # 'signed' (порог) | 'gauss' (колокол)\n    enc_thr = 0.25\n    enc_span = 0.15\n    tuning_sigma = 0.35\n    i_sens_max = 22.0                # мкА/см2\n\n    # ---- шум и смещения\n    hid_bias = 0.0\n    noise_sigma = 0.5                # фоновый шум (скрытые)\n    mot_bias = 0.0                   # моторные при ОЦЕНКЕ\n    mot_noise = 0.5\n    explore_noise = 2.0              # моторные при ОБУЧЕНИИ (исследование); None = как mot_noise\n    explore_bias = 0.0\n    explore_hid_noise = None         # шум скрытых при ОБУЧЕНИИ (нужен для обучения скрытого слоя); None = как noise_sigma\n    feedback_noise = False           # шум-штраф на сенсорах после промаха (идея DishBrain)\n\n    # ---- синапсы\n    g_scale = 0.3                    # мСм/см2 на единицу веса W\n    w_max = 0.8\n    w_in_cap = 1.6\n    syn_tau_e, syn_tau_i = 5.0, 10.0\n\n    # ---- обучение (RM-Hebb: dW = lr * da * pre * (post - <post>))\n    rule = \"rmhebb\"                  # 'rmhebb' (рекомендуется) | 'legacy' (как в v2.1, для сравнения)\n    lr = 0.15\n    lr_hid_mult = 1.0                # множитель lr для синапсов НА скрытые нейроны\n    e_decay = 0.0                    # память следа между кадрами (0 = только текущий кадр)\n    pre_cur_w = 1.0                  # вес пресинаптических спайков ТЕКУЩЕГО кадра\n    pre_prev_w = 0.0                 # вес спайков ПРОШЛОГО кадра\n    shaping = 4.0                    # награда за сокращение ошибки движением ракетки\n    in_window_bonus = 0.0\n    r_hit, r_miss = 0.0, 0.0\n    use_baseline = False             # вычитать скользящую среднюю награды (в v2.1 было True: давало дрейф весов)\n    da_eps = 1e-6                    # |da| ниже порога: обновления нет (защита от дрейфа в паузах)\n    pavg_tau = 0.02\n\n    # ---- структурная пластичность\n    structural = False\n    struct_every = 10\n    grow_below = 0.95\n    prune_w = 0.01\n    grace_episodes = 60              # защита новорождённых от апоптоза\n\n    # ---- химико-термическая среда (BioEnvironment)\n    env_enabled = True\n    temperature = 37.0               # °C\n    temp_ref = 37.0                  # температура, при которой калиброваны параметры HH\n    q10_gate = 3.0                   # кинетика воротных переменных m, h, n\n    q10_cond = 1.4                   # максимальные проводимости ионных каналов\n    q10_syn = 2.0                    # кинетика синапса (скорость спада)\n    # пулы нейромедиаторов, относительные единицы (1.0 = референсный фон)\n    glu_tonic, gaba_tonic, da_tonic = 1.0, 1.0, 1.0\n    glu_release, gaba_release = 0.002, 0.002     # прирост за один спайк\n    da_release = 0.15                # прирост фазического дофамина на единицу |награды|\n    glu_tau, gaba_tau, da_tau = 30.0, 50.0, 80.0  # мс, клиренс (обратный захват)\n    env_k = 1.0                      # полунасыщение функции Хилла\n    da_gate_plasticity = True        # дофамин управляет скоростью обучения\n\n\nPRESETS = {\n    \"direct\": dict(n_hid0=0, direct_p=1.0, structural=False),\n    \"hidden\": dict(n_hid0=14, direct_p=0.5, structural=False),\n    \"hidden_structural\": dict(n_hid0=14, direct_p=0.5, structural=True),\n    \"two_layer\": dict(n_hid0=24, direct_p=0.0, structural=False, explore_hid_noise=2.0),\n    \"legacy_v21\": dict(n_hid0=0, direct_p=1.0, rule=\"legacy\", e_decay=0.8, pre_prev_w=1.0, use_baseline=True,\n                       in_window_bonus=0.1, r_hit=0.6, r_miss=-0.6, env_enabled=False, feedback_noise=True),\n}\n\n\ndef make_config(preset=\"direct\", **kw):\n    \"\"\"Config с пресетом и переопределениями. Опечатка в имени параметра -> ошибка.\"\"\"\n    c = Config()\n    for k, v in {**PRESETS[preset], **kw}.items():\n        if not hasattr(c, k):\n            raise KeyError(f\"Неизвестный параметр Config: {k}\")\n        setattr(c, k, v)\n    return c\n\n\ndef cfg_to_dict(cfg):\n    d = {}\n    for k in dir(Config):\n        if k.startswith(\"_\") or callable(getattr(Config, k)):\n            continue\n        v = getattr(cfg, k)\n        d[k] = list(v) if isinstance(v, tuple) else v\n    return d\n\n\ndef cfg_from_dict(d):\n    c = Config()\n    for k, v in d.items():\n        if hasattr(c, k):\n            setattr(c, k, tuple(v) if isinstance(getattr(Config, k), tuple) else v)\n    return c\n\n\ndef save_config(cfg, path=\"experiment_cfg.json\"):\n    json.dump(cfg_to_dict(cfg), open(path, \"w\"), ensure_ascii=False, indent=1)\n\n\ndef load_config(path=\"experiment_cfg.json\"):\n    return cfg_from_dict(json.load(open(path)))\n\n\ndef log_run(record, path=\"oi_runs.jsonl\"):\n    rec = dict(record)\n    rec[\"timestamp\"] = time.strftime(\"%Y-%m-%d %H:%M:%S\")\n    with open(path, \"a\", encoding=\"utf-8\") as f:\n        f.write(json.dumps(rec, ensure_ascii=False, default=str) + \"\\n\")\n", "environment.py": "\"\"\"BioEnvironment: температура и химическая среда (глутамат, ГАМК, дофамин).\n\nТемпература масштабирует кинетику через Q10:   фактор(T) = Q10 ** ((T - T_ref) / 10)\n  - воротные переменные m, h, n (Q10 = 3):  dm/dt умножается на фактор (спайки быстрее при нагреве);\n  - проводимости каналов (Q10 ~ 1.4):        g_Na, g_K, g_L умножаются на фактор;\n  - синапс (Q10 ~ 2):                         время спада tau делится на фактор.\nПараметры HH считаются откалиброванными при T_ref (по умолчанию 37 °C). Классические HH-параметры\nполучены при 6.3 °C; чтобы их использовать «как есть», поставьте temp_ref=6.3.\n\nНейромедиаторы: относительные концентрации (1.0 = референсный фон). Каждый пул:\n    C(t+dt) = tonic + (C - tonic) * exp(-dt / tau) + выброс\nВыброс: глутамат от возбуждающих спайков, ГАМК от тормозных, фазический дофамин от |награды|.\nВлияние (функция Хилла, нормированная так, что при C = 1 множитель равен 1, при C = 0 равен 0):\n    gain(C) = 2*C / (C + 1)      (при env_k = 1; ограничена сверху значением 2)\n  - глутамат -> усиление возбуждающих синапсов;  ГАМК -> усиление тормозных;\n  - дофамин  -> множитель скорости пластичности lr (при 0 обучение выключено).\nПространственной диффузии нет: пулы «хорошо перемешаны» (расширение: сетка вокселей).\n\"\"\"\nimport numpy as np\n\n\nclass BioEnvironment:\n    def __init__(self, cfg):\n        self.cfg = cfg\n        self.T = float(cfg.temperature)\n        self.tonic = dict(glu=cfg.glu_tonic, gaba=cfg.gaba_tonic, da=cfg.da_tonic)\n        self.level = dict(self.tonic)\n        self.tau = dict(glu=cfg.glu_tau, gaba=cfg.gaba_tau, da=cfg.da_tau)\n        self.history = []\n\n    # ---------------- температура\n    def _q(self, q10):\n        return float(q10 ** ((self.T - self.cfg.temp_ref) / 10.0))\n\n    @property\n    def kin_factor(self):      # воротные переменные\n        return self._q(self.cfg.q10_gate) if self.cfg.env_enabled else 1.0\n\n    @property\n    def cond_factor(self):     # проводимости\n        return self._q(self.cfg.q10_cond) if self.cfg.env_enabled else 1.0\n\n    @property\n    def syn_rate_factor(self):  # скорость спада синапса\n        return self._q(self.cfg.q10_syn) if self.cfg.env_enabled else 1.0\n\n    # ---------------- химия\n    def _gain(self, c):\n        k = self.cfg.env_k\n        ref = 1.0 / (1.0 + k)\n        return (max(c, 0.0) / (max(c, 0.0) + k)) / ref if self.cfg.env_enabled else 1.0\n\n    @property\n    def glu_gain(self):\n        return self._gain(self.level[\"glu\"])\n\n    @property\n    def gaba_gain(self):\n        return self._gain(self.level[\"gaba\"])\n\n    @property\n    def plasticity_gain(self):\n        if not (self.cfg.env_enabled and self.cfg.da_gate_plasticity):\n            return 1.0\n        return self._gain(self.level[\"da\"])\n\n    def set_temperature(self, T):\n        self.T = float(T)\n\n    def set_tonic(self, name, value):\n        \"\"\"Например env.set_tonic('da', 0.2): истощение дофамина (обучение почти выключено).\"\"\"\n        self.tonic[name] = float(value)\n\n    def step(self, dt_ms, n_exc_spikes=0.0, n_inh_spikes=0.0, reward=0.0):\n        \"\"\"Обновить пулы за интервал dt_ms (обычно один кадр).\"\"\"\n        c = self.cfg\n        for k in self.level:\n            self.level[k] = self.tonic[k] + (self.level[k] - self.tonic[k]) * np.exp(-dt_ms / self.tau[k])\n        self.level[\"glu\"] += c.glu_release * n_exc_spikes\n        self.level[\"gaba\"] += c.gaba_release * n_inh_spikes\n        self.level[\"da\"] += c.da_release * abs(reward)\n\n    def snapshot(self):\n        return dict(T=float(self.T), kin=round(float(self.kin_factor), 4), Glu=round(float(self.level[\"glu\"]), 4),\n                    GABA=round(float(self.level[\"gaba\"]), 4), DA=round(float(self.level[\"da\"]), 4),\n                    plasticity_gain=round(float(self.plasticity_gain), 4))\n\n    # для сохранения\n    def state(self):\n        return dict(T=self.T, tonic=dict(self.tonic), level=dict(self.level))\n\n    def load(self, d):\n        self.T, self.tonic, self.level = d[\"T\"], dict(d[\"tonic\"]), dict(d[\"level\"])\n", "experiments.py": "\"\"\"Измерения: базовые линии, оракул, бенчмарк по seed, абляции, ступенчатая отладка.\"\"\"\nimport numpy as np\nimport pandas as pd\nfrom .config import make_config, ROLE_SENS\nfrom .sim import OIPong\n\n\ndef baseline_winrate(cfg, policy=\"random\", n=2000, seed=0):\n    \"\"\"Win rate БЕЗ сети: 'random' | 'stay' | 'perfect'. Физика берётся из cfg.\"\"\"\n    rng = np.random.default_rng(seed)\n    hits = 0\n    for _ in range(n):\n        by, vy = rng.uniform(0.1, 0.9), rng.choice([-1, 1]) * rng.uniform(0.0, 0.04)\n        py, bx = rng.uniform(0.2, 0.8), 0.0\n        while True:\n            mv = cfg.paddle_gain * rng.integers(-3, 4) if policy == \"random\" else (by - py if policy == \"perfect\" else 0.0)\n            py = float(np.clip(py + np.clip(mv, -cfg.paddle_max, cfg.paddle_max), 0.0, 1.0))\n            bx += cfg.ball_vx\n            by += vy\n            if by < 0.0 or by > 1.0:\n                vy *= -1.0\n                by = float(np.clip(by, 0.0, 1.0))\n            if bx >= 1.0:\n                hits += abs(by - py) <= cfg.hit_half\n                break\n    return hits / n\n\n\ndef make_oracle(cfg, w=None):\n    \"\"\"Сеть с вручную выставленными весами (сенсор pref>0 -> UP, pref<0 -> DOWN): потолок считывания.\"\"\"\n    g = OIPong(cfg)\n    w = cfg.w_max if w is None else w\n    g.mask[:] = False\n    g.W[:] = 0.0\n    for s in g.idx(ROLE_SENS):\n        p = g.pref[s]\n        if abs(p) < 0.05:\n            continue\n        tgt = g.mot_up if p > 0 else g.mot_down\n        g.mask[s, tgt] = True\n        g.W[s, tgt] = w\n    return g\n\n\ndef weight_structure(g):\n    \"\"\"Средний вес «верных» (pref>0 -> UP, pref<0 -> DOWN) и «неверных» прямых путей. Должны расходиться.\"\"\"\n    S = g.idx(ROLE_SENS)\n    up, dn = g.mot_up, g.mot_down\n    pos, neg = S[g.pref[S] > 0], S[g.pref[S] < 0]\n    ok = (g.W[np.ix_(pos, up)].mean() + g.W[np.ix_(neg, dn)].mean()) / 2\n    bad = (g.W[np.ix_(pos, dn)].mean() + g.W[np.ix_(neg, up)].mean()) / 2\n    return float(ok), float(bad)\n\n\ndef benchmark(seeds=(1, 2, 3), episodes=1500, eval_every=300, eval_n=100, preset=\"direct\", verbose=True, **cfg_kw):\n    rows = []\n    for sd in seeds:\n        g = OIPong(make_config(preset, seed=sd, **cfg_kw))\n        _, _, ev, _ = g.train(target=None, max_episodes=episodes, eval_every=eval_every, eval_n=eval_n,\n                              frame_log_every=0, verbose=False)\n        for _, r in ev.iterrows():\n            rows.append(dict(Seed=sd, Episode=int(r[\"Episode\"]), Eval_Win_Pct=float(r[\"Eval_Win_Pct\"]),\n                             Neurons_Total=int(r[\"Neurons_Total\"])))\n        if verbose:\n            print(f\"seed {sd}: \" + \" \".join(f\"{int(r.Episode)}:{r.Eval_Win_Pct:.0f}%\" for r in ev.itertuples()))\n    df = pd.DataFrame(rows)\n    return df, df.groupby(\"Episode\")[\"Eval_Win_Pct\"].agg([\"mean\", \"std\", \"min\", \"max\"]).round(1)\n\n\ndef ablation(variants, seeds=(1, 2, 3), episodes=1500, eval_every=300, eval_n=100, verbose=True):\n    \"\"\"variants: {имя: dict(preset=..., параметры Config...)}. Возвращает mean eval по вариантам.\"\"\"\n    out = {}\n    for name, kw in variants.items():\n        kw = dict(kw)\n        preset = kw.pop(\"preset\", \"direct\")\n        if verbose:\n            print(f\"== {name}\")\n        _, summ = benchmark(seeds=seeds, episodes=episodes, eval_every=eval_every, eval_n=eval_n, preset=preset,\n                            verbose=verbose, **kw)\n        out[name] = summ[\"mean\"]\n    return pd.DataFrame(out)\n\n\ndef debug_ladder(episodes=300, eval_n=100, seed=1, verbose=True):\n    \"\"\"Пошаговая отладка: каждый следующий шаг добавляет одну сложность. Читать сверху вниз; шаг, на котором\n    обучение ломается, показывает виновника.\n       1 оракул (потолок считывания)  2 direct + обучение  3 + скрытый слой  4 + нейрогенез\n       5 + температура/химия при норме  6 нагрев/охлаждение (скорость дофамина мерьте через dopamine_scan)\"\"\"\n    steps = [\n        (\"1 оракул (потолок)\", None, \"direct\", {}),\n        (\"2 direct, обучение (среда ВЫКЛ)\", \"learn\", \"direct\", dict(env_enabled=False)),\n        (\"3 + скрытый слой\", \"learn\", \"hidden\", {}),\n        (\"4 + нейрогенез\", \"learn\", \"hidden_structural\", {}),\n        (\"5 + среда ВКЛ (37 °C, норма)\", \"learn\", \"direct\", dict(env_enabled=True)),\n        (\"6a T = 32 °C\", \"learn\", \"direct\", dict(temperature=32.0)),\n        (\"6b T = 41 °C\", \"learn\", \"direct\", dict(temperature=41.0)),\n    ]\n    rows = []\n    for name, mode, preset, kw in steps:\n        cfg = make_config(preset, seed=seed, **kw)\n        if mode is None:\n            g = make_oracle(cfg)\n            win = g.evaluate(eval_n)[\"win\"] * 100\n            rows.append(dict(Шаг=name, До=None, После=round(win, 1), Верные_W=None, Неверные_W=None))\n        else:\n            g = OIPong(cfg)\n            before = g.evaluate(eval_n)[\"win\"] * 100\n            g.train(target=None, max_episodes=episodes, eval_every=0, frame_log_every=0, verbose=False)\n            after = g.evaluate(eval_n)[\"win\"] * 100\n            ok, bad = weight_structure(g) if len(g.idx(ROLE_SENS)) and g.mask[np.ix_(g.idx(ROLE_SENS), g.mot_up)].any() else (None, None)\n            rows.append(dict(Шаг=name, До=round(before, 1), После=round(after, 1),\n                             Верные_W=None if ok is None else round(ok, 3), Неверные_W=None if bad is None else round(bad, 3)))\n        if verbose:\n            print(rows[-1], flush=True)\n    return pd.DataFrame(rows)\n\n\ndef temperature_scan(temps=(25.0, 30.0, 34.0, 37.0, 40.0, 42.0), n=60, seed=1):\n    \"\"\"Как температура меняет СЕТЬ (Q10): частота спайков сенсоров/моторов при фиксированном входе и\n    win rate оракула. Чем выше T, тем быстрее кинетика каналов -> больше спайков за кадр.\"\"\"\n    rows = []\n    for T in temps:\n        cfg = make_config(\"direct\", seed=seed, temperature=T)\n        g = make_oracle(cfg)\n        sens, mot = [], []\n        for _ in range(30):\n            g.game.reset(g.rng)\n            g.game.ball_y, g.game.paddle_y = 0.8, 0.3      # большая положительная ошибка\n            sp = g.step(learn=False)[\"spikes\"]\n            sens.append(sp[g.idx(ROLE_SENS)].sum())\n            mot.append(sp[g.mot_up].sum())\n        rows.append(dict(T_C=T, kin_factor=round(g.env.kin_factor, 3), Spikes_Sens=round(float(np.mean(sens)), 2),\n                         Spikes_MotUP=round(float(np.mean(mot)), 2), Oracle_Win_Pct=round(100 * g.evaluate(n)[\"win\"], 1)))\n    return pd.DataFrame(rows)\n\n\ndef dopamine_scan(levels=(1.0, 0.3, 0.1, 0.0), episodes=60, eval_n=60, seed=1, preset=\"direct\"):\n    \"\"\"Скорость обучения при разном тоническом дофамине (ранний eval, а не итоговый: итог часто 100 % у всех).\n    При da_tonic=0 пластичность выключена (множитель lr = 0): сеть должна остаться на уровне необученной.\"\"\"\n    rows = []\n    for lv in levels:\n        g = OIPong(make_config(preset, seed=seed, da_tonic=lv, da_release=0.0 if lv == 0 else 0.15))\n        before = g.evaluate(eval_n)[\"win\"] * 100\n        g.train(target=None, max_episodes=episodes, eval_every=0, frame_log_every=0, verbose=False)\n        ok, bad = weight_structure(g)\n        rows.append(dict(DA_tonic=lv, Gain_lr=round(g.env.plasticity_gain, 3), Eval_До=round(before, 1),\n                         Eval_После=round(g.evaluate(eval_n)[\"win\"] * 100, 1), Верные_W=round(ok, 3), Неверные_W=round(bad, 3)))\n    return pd.DataFrame(rows)\n", "game.py": "\"\"\"Физика Pong и кодирование наблюдения. Ничего не знает о нейронах.\"\"\"\nimport numpy as np\n\n\nclass PongGame:\n    def __init__(self, cfg):\n        self.cfg = cfg\n        self.ball_x = self.ball_y = self.ball_vy = 0.0\n        self.paddle_y = 0.5\n\n    def reset(self, rng):\n        self.ball_x = 0.0\n        self.ball_y = float(rng.uniform(0.1, 0.9))\n        self.ball_vy = float(rng.choice([-1, 1]) * rng.uniform(0.0, 0.04))\n        self.paddle_y = float(rng.uniform(0.2, 0.8))\n\n    @property\n    def error(self):\n        return self.ball_y - self.paddle_y\n\n    def encode(self, pref):\n        \"\"\"Активность каждого сенсорного нейрона [0..1] по ошибке мяч - ракетка.\"\"\"\n        c, err = self.cfg, self.error\n        if c.encoding == \"signed\":\n            p = np.nan_to_num(pref)\n            return np.clip((np.sign(p) * err - c.enc_thr * np.abs(p)) / c.enc_span, 0.0, 1.0)\n        return np.exp(-0.5 * ((err - pref) / c.tuning_sigma) ** 2)\n\n    def advance(self, move):\n        \"\"\"Двигает ракетку и мяч на один кадр. Возвращает (paddle_old, done, hit).\"\"\"\n        c = self.cfg\n        paddle_old = self.paddle_y\n        self.paddle_y = float(np.clip(self.paddle_y + move, 0.0, 1.0))\n        self.ball_x += c.ball_vx\n        self.ball_y += self.ball_vy\n        if self.ball_y < 0.0 or self.ball_y > 1.0:\n            self.ball_vy *= -1.0\n            self.ball_y = float(np.clip(self.ball_y, 0.0, 1.0))\n        done = self.ball_x >= 1.0\n        hit = bool(done and abs(self.error) <= c.hit_half)\n        return paddle_old, done, hit\n\n    def shaping_reward(self, paddle_old):\n        \"\"\"Эффект ИМЕННО движения ракетки: 0, если ракетка не двигалась (естественный baseline).\"\"\"\n        c = self.cfg\n        return c.shaping * (abs(self.ball_y - paddle_old) - abs(self.error)) + \\\n            c.in_window_bonus * float(abs(self.error) <= c.hit_half)\n", "hh.py": "\"\"\"Нейронная биофизика: Ходжкин-Хаксли (таблицы скоростей) + синапсы-проводимости.\nТемпература и нейромедиаторы приходят из BioEnvironment (kin_factor, cond_factor, glu/gaba gain).\"\"\"\nimport numpy as np\n\n_VMIN, _VMAX, _RES = -120.0, 80.0, 20.0\n_VG = np.arange(_VMIN, _VMAX + 1.0 / _RES, 1.0 / _RES)\n\n\ndef _vtrap(x, y):\n    x = np.where(np.abs(x) < 1e-7, 1e-7, x)\n    return x / (-np.expm1(-x / y))\n\n\n_a_m = 0.1 * _vtrap(_VG + 40.0, 10.0)\n_b_m = 4.0 * np.exp(-(_VG + 65.0) / 18.0)\n_a_h = 0.07 * np.exp(-(_VG + 65.0) / 20.0)\n_b_h = 1.0 / (1.0 + np.exp(-(_VG + 35.0) / 10.0))\n_a_n = 0.01 * _vtrap(_VG + 55.0, 10.0)\n_b_n = 0.125 * np.exp(-(_VG + 65.0) / 80.0)\n_S_m, _S_h, _S_n = _a_m + _b_m, _a_h + _b_h, _a_n + _b_n\n\nG_NA, E_NA = 120.0, 50.0\nG_K, E_K = 36.0, -77.0\nG_L, E_L = 0.3, -54.387\nE_EXC, E_INH = 0.0, -80.0\nREST = dict(v=-65.0, m=0.053, h=0.596, n=0.318)\n\n\ndef integrate_frame(net, cfg, I_ext, rng, env, explore=False):\n    \"\"\"Один кадр (frame_ms): интегрирует все нейроны. Возвращает число спайков каждого нейрона.\"\"\"\n    from .config import ROLE_SENS, ROLE_HID, ROLE_MOT\n    S = int(round(cfg.frame_ms / cfg.dt))\n    N = net.N\n    mn = cfg.explore_noise if (explore and cfg.explore_noise is not None) else cfg.mot_noise\n    mb = cfg.explore_bias if (explore and cfg.explore_bias is not None) else cfg.mot_bias\n    hn = cfg.explore_hid_noise if (explore and cfg.explore_hid_noise is not None) else cfg.noise_sigma\n    noise = rng.normal(0.0, 1.0, (S // 20 + 1, N))\n    scale = np.full(N, cfg.noise_sigma)\n    scale[net.role == ROLE_HID] = hn\n    scale[net.role == ROLE_MOT] = mn if mn is not None else cfg.noise_sigma\n    scale[net.role == ROLE_SENS] = 0.0\n    noise *= scale\n    bias = np.where(net.role == ROLE_HID, cfg.hid_bias, 0.0) + np.where(net.role == ROLE_MOT, 0.0 if mb is None else mb, 0.0)\n\n    phi = env.kin_factor                    # температурный фактор кинетики m, h, n\n    gk = env.cond_factor                    # температурный фактор проводимостей\n    v, m, h, n = net.v, net.m, net.h, net.n\n    ge, gi, above = net.ge, net.gi, net.above\n    counts = np.zeros(N)\n    dt = cfg.dt\n    sr = env.syn_rate_factor\n    de, di = np.exp(-dt * sr / cfg.syn_tau_e), np.exp(-dt * sr / cfg.syn_tau_i)\n    gs = cfg.g_scale\n    We = gs * env.glu_gain * net.W * (~net.inhib)[:, None]       # возбуждение: глутамат\n    Wi = gs * env.gaba_gain * net.W * net.inhib[:, None]         # торможение: ГАМК\n    gna, gkk, gl = G_NA * gk, G_K * gk, G_L * gk\n    top = len(_VG) - 1\n    for s in range(S):\n        idx = np.clip(((v - _VMIN) * _RES).astype(int), 0, top)\n        m += dt * phi * (_a_m[idx] - _S_m[idx] * m)\n        h += dt * phi * (_a_h[idx] - _S_h[idx] * h)\n        n += dt * phi * (_a_n[idx] - _S_n[idx] * n)\n        i_ion = gna * m ** 3 * h * (v - E_NA) + gkk * n ** 4 * (v - E_K) + gl * (v - E_L)\n        i_syn = ge * (E_EXC - v) + gi * (E_INH - v)\n        v += dt * (I_ext + bias + noise[s // 20] + i_syn - i_ion)\n        np.clip(v, -100.0, 70.0, out=v)\n        ge *= de\n        gi *= di\n        now = v > 0.0\n        sp = now & ~above\n        above[:] = now\n        if sp.any():\n            counts += sp\n            spf = sp.astype(float)\n            ge += spf @ We\n            gi += spf @ Wi\n    return counts\n", "learning.py": "\"\"\"Правило обучения RM-Hebb (дофамин-модулируемая пластичность).\n\n    E  <- e_decay * E + outer(pre, post - <post>)          # след приоритета\n    W  <- clip(W + lr_eff * da * E)                        # da = награда (- baseline, если use_baseline)\n\nПочему это работает там, где v2.1 не работало:\n  1. da = награда за эффект ДВИЖЕНИЯ ракетки: при «не двигался» da = 0 -> нулевой baseline, и веса\n     НЕ дрейфуют (в v2.1 скользящее среднее награды делало da != 0 в каждом кадре -> дрейф);\n  2. награда приходит в том же кадре, что и действие, поэтому след берёт спайки ТЕКУЩЕГО кадра и не\n     хранит прошлое (e_decay = 0): иначе вина за действие размазывается на соседние кадры;\n  3. пост-активность центрируется (post - <post>): случайные колебания дофамина вычитаются,\n     остаётся ковариация «шум исследования <-> награда» (оценка градиента, REINFORCE).\n\"\"\"\nimport numpy as np\nfrom .config import ROLE_HID\n\n\ndef apply_plasticity(net, cfg, spikes, prev_spk, da, lr_gain=1.0):\n    dev = spikes - net.pavg\n    net.pavg = (1.0 - cfg.pavg_tau) * net.pavg + cfg.pavg_tau * spikes\n    pre = cfg.pre_cur_w * spikes + cfg.pre_prev_w * prev_spk\n    net.E = cfg.e_decay * net.E + np.outer(pre, dev) * net.mask\n    if abs(da) < cfg.da_eps or lr_gain <= 0.0:\n        return\n    lr = np.full(net.N, cfg.lr * lr_gain)\n    lr[net.role == ROLE_HID] *= cfg.lr_hid_mult          # скорость для синапсов НА скрытые нейроны\n    net.W += lr[None, :] * da * net.E\n    net.W = np.clip(net.W, 0.0, cfg.w_max) * net.mask\n    net.scale_inputs()\n", "network.py": "\"\"\"Структура сети: 3D-геометрия, роли нейронов, связность, нейрогенез/апоптоз, прунинг/спраутинг.\nРоли хранятся в массиве role (а не в списках индексов), поэтому вход/выход не «съезжают» при удалении.\"\"\"\nimport numpy as np\nimport pandas as pd\nfrom .config import ROLE_SENS, ROLE_HID, ROLE_MOT, ROLE_NAME\nfrom .hh import REST\n\nPER_NEURON = [\"xyz\", \"v\", \"m\", \"h\", \"n\", \"ge\", \"gi\", \"role\", \"nid\", \"inhib\", \"rate\", \"above\", \"pavg\", \"born\",\n              \"mgroup\", \"pref\"]\nPER_PAIR = [\"W\", \"E\", \"mask\", \"C\"]\n\n\nclass Network:\n    def __init__(self, cfg, rng):\n        self.cfg = cfg\n        self.rng = rng\n        self.box = np.array(cfg.box)\n        self.next_id = 0\n        self.episode = 0\n        self.events = []\n        self._build()\n\n    # ------------------------------------------------------------ базовое\n    @property\n    def N(self):\n        return len(self.v)\n\n    def idx(self, role):\n        return np.where(self.role == role)[0]\n\n    @property\n    def mot_up(self):\n        return np.where((self.role == ROLE_MOT) & (self.mgroup == 0))[0]\n\n    @property\n    def mot_down(self):\n        return np.where((self.role == ROLE_MOT) & (self.mgroup == 1))[0]\n\n    def _alloc(self, n):\n        self.xyz = np.zeros((n, 3))\n        self.v = np.full(n, REST[\"v\"])\n        self.m = np.full(n, REST[\"m\"])\n        self.h = np.full(n, REST[\"h\"])\n        self.n = np.full(n, REST[\"n\"])\n        self.ge, self.gi = np.zeros(n), np.zeros(n)\n        self.role = np.zeros(n, dtype=int)\n        self.nid = np.zeros(n, dtype=int)\n        self.inhib = np.zeros(n, dtype=bool)\n        self.rate = np.zeros(n)\n        self.above = np.zeros(n, dtype=bool)\n        self.pavg = np.full(n, 0.3)\n        self.born = np.zeros(n, dtype=int)\n        self.mgroup = np.full(n, -1)\n        self.pref = np.full(n, np.nan)\n        self.W = np.zeros((n, n))\n        self.E = np.zeros((n, n))\n        self.C = np.zeros((n, n))\n        self.mask = np.zeros((n, n), dtype=bool)\n\n    def _build(self):\n        c, rng, box = self.cfg, self.rng, self.box\n        ns, nh, nm = c.n_sens, c.n_hid0, 2 * c.n_mot_each\n        n = ns + nh + nm\n        self._alloc(n)\n        placed = []\n\n        def place(lo_x, hi_x):\n            p = None\n            for _ in range(500):\n                p = np.array([rng.uniform(lo_x, hi_x), rng.uniform(20, box[1] - 20), rng.uniform(20, box[2] - 20)])\n                if not placed or np.min(np.linalg.norm(np.array(placed) - p, axis=1)) >= 40.0:\n                    break\n            placed.append(p)\n            return p\n\n        prefs = np.linspace(-0.85, 0.85, ns)\n        for i in range(n):\n            if i < ns:\n                self.role[i] = ROLE_SENS\n                self.xyz[i] = place(15, 70)\n                self.pref[i] = prefs[i]\n            elif i < ns + nh:\n                self.role[i] = ROLE_HID\n                self.xyz[i] = place(90, box[0] - 90)\n            else:\n                self.role[i] = ROLE_MOT\n                self.xyz[i] = place(box[0] - 70, box[0] - 15)\n                self.mgroup[i] = 0 if i < ns + nh + c.n_mot_each else 1\n            self.nid[i] = self.next_id\n            self.next_id += 1\n        self.inhib = (self.role == ROLE_HID) & (rng.random(n) < c.inhib_frac)\n        self._connect_initial()\n\n    def allowed(self):\n        a = np.outer(self.role != ROLE_MOT, self.role != ROLE_SENS)\n        np.fill_diagonal(a, False)\n        return a\n\n    def dist(self):\n        return np.linalg.norm(self.xyz[:, None, :] - self.xyz[None, :, :], axis=2)\n\n    def _connect_initial(self):\n        c, rng = self.cfg, self.rng\n        d, allowed = self.dist(), self.allowed()\n        p = np.clip(np.exp(-d / c.conn_lambda) * (d <= c.conn_radius) * allowed * 1.6, 0, 0.9)\n        mask = (rng.random(p.shape) < p) & allowed\n        if self.idx(ROLE_HID).size:\n            for i in self.idx(ROLE_SENS):\n                cand = np.where(allowed[i] & (self.role == ROLE_HID))[0]\n                mask[i, cand[np.argsort(d[i, cand])][:c.min_deg]] = True\n            for j in self.idx(ROLE_MOT):\n                cand = np.where(allowed[:, j] & (self.role == ROLE_HID))[0]\n                mask[cand[np.argsort(d[cand, j])][:c.min_deg], j] = True\n        # длинные проекции входного электродного массива прямо в моторную зону\n        mask &= ~np.outer(self.role == ROLE_SENS, self.role == ROLE_MOT)\n        sm = np.outer(self.role == ROLE_SENS, self.role == ROLE_MOT)\n        mask |= sm & (rng.random(mask.shape) < c.direct_p)\n        if c.direct_p <= 0:      # чисто многослойная сеть: никаких коротких путей сенсор -> мотор\n            mask &= ~sm\n        self.mask = mask\n        self.W = np.where(mask, rng.uniform(c.w_init[0], c.w_init[1], mask.shape), 0.0)\n        self.scale_inputs()\n\n    def scale_inputs(self):\n        \"\"\"Синаптическое масштабирование: сумма входящих возбуждающих весов <= w_in_cap.\"\"\"\n        exc = ~self.inhib\n        tot = (self.W * exc[:, None]).sum(axis=0)\n        over = tot > self.cfg.w_in_cap\n        if over.any():\n            f = np.where(over, self.cfg.w_in_cap / np.maximum(tot, 1e-9), 1.0)\n            self.W[exc, :] *= f[None, :]\n\n    # ------------------------------------------------- структурная пластичность\n    def _append(self, xyz, role, inhib, mgroup=-1, pref=np.nan):\n        n = self.N\n        fills = dict(v=REST[\"v\"], m=REST[\"m\"], h=REST[\"h\"], n=REST[\"n\"], ge=0.0, gi=0.0, role=role,\n                     nid=self.next_id, inhib=inhib, rate=0.0, above=False, pavg=0.3, born=self.episode,\n                     mgroup=mgroup, pref=pref)\n        self.xyz = np.vstack([self.xyz, xyz])\n        for k, val in fills.items():\n            setattr(self, k, np.append(getattr(self, k), val))\n        for k in PER_PAIR:\n            a = getattr(self, k)\n            new = np.zeros((n + 1, n + 1), dtype=a.dtype)\n            new[:n, :n] = a\n            setattr(self, k, new)\n        self.next_id += 1\n        return n\n\n    def _remove(self, i):\n        for k in PER_NEURON:\n            setattr(self, k, np.delete(getattr(self, k), i, axis=0))\n        for k in PER_PAIR:\n            setattr(self, k, np.delete(np.delete(getattr(self, k), i, axis=0), i, axis=1))\n\n    def log_event(self, kind, nid, note):\n        self.events.append(dict(Episode=self.episode, Event=kind, Neuron_ID=nid, Note=note,\n                                Hidden=int((self.role == ROLE_HID).sum()), Total=self.N))\n\n    def neurogenesis(self):\n        hid = self.idx(ROLE_HID)\n        if len(hid) == 0 or len(hid) >= self.cfg.hid_max:\n            return None\n        util = self.rate[hid] * (self.W[hid, :].sum(axis=1) + 0.05)\n        parent = hid[int(np.argmax(util + 1e-6 * self.rng.random(len(hid))))]\n        xyz = np.clip(self.xyz[parent] + self.rng.normal(0, 18.0, 3), 10, self.box - 10)\n        child = self._append(xyz, ROLE_HID, bool(self.inhib[parent]))\n        self.rate[child] = 0.5 * self.rate[parent]\n        for j in range(child):\n            if self.mask[parent, j]:\n                self.mask[child, j] = True\n                self.W[child, j] = self.W[parent, j] * self.rng.uniform(0.7, 1.2)\n            if self.mask[j, parent]:\n                self.mask[j, child] = True\n                self.W[j, child] = self.W[j, parent] * self.rng.uniform(0.7, 1.2)\n        self.mask &= self.allowed()\n        self.W *= self.mask\n        self.scale_inputs()\n        self.log_event(\"neurogenesis\", int(self.nid[child]), f\"родитель {int(self.nid[parent])}\")\n        return child\n\n    def apoptosis(self):\n        hid = self.idx(ROLE_HID)\n        if len(hid) <= self.cfg.hid_min:\n            return None\n        hid = hid[(self.episode - self.born[hid]) >= self.cfg.grace_episodes]\n        if len(hid) == 0:\n            return None\n        util = self.rate[hid] * (self.W[hid, :].sum(axis=1) + 0.05) + 0.02 * self.W[:, hid].sum(axis=0)\n        victim = hid[int(np.argmin(util))]\n        nid = int(self.nid[victim])\n        self._remove(victim)\n        self.log_event(\"apoptosis\", nid, \"низкая полезность\")\n        return victim\n\n    def structural_update(self, rolling_win):\n        c = self.cfg\n        weak = self.mask & (self.W < c.prune_w)\n        if weak.any():\n            n_weak = int(weak.sum())\n            self.mask[weak] = False\n            self.W[weak] = 0.0\n            self.log_event(\"synapse_prune\", -1, f\"{n_weak} синапсов\")\n        allowed = self.allowed() & ~self.mask & (self.dist() <= c.conn_radius)\n        cand = np.where(allowed, self.C, -1.0)\n        for _ in range(3):\n            k = np.unravel_index(np.argmax(cand), cand.shape)\n            if cand[k] <= 1e-4:\n                break\n            self.mask[k] = True\n            self.W[k] = 0.08\n            cand[k] = -1.0\n            self.log_event(\"synapse_sprout\", int(self.nid[k[0]]), f\"-> {int(self.nid[k[1]])}\")\n        if rolling_win < c.grow_below:\n            self.neurogenesis()\n        hid = self.idx(ROLE_HID)\n        if len(hid) > c.hid_min and (self.rate[hid] < 0.01).any():\n            self.apoptosis()\n        self.rate *= 0.5\n        self.scale_inputs()\n\n    # ---------------------------------------------------------- таблицы\n    def class_w(self, pre_role, post_role):\n        m = self.mask & np.outer(self.role == pre_role, self.role == post_role)\n        return float(self.W[m].mean()) if m.any() else 0.0\n\n    def mean_w(self):\n        return float(self.W[self.mask].mean()) if self.mask.any() else 0.0\n\n    def neurons_table(self):\n        return pd.DataFrame(dict(Neuron_ID=self.nid, Role=[ROLE_NAME[r] for r in self.role],\n                                 Inhibitory=self.inhib, X_um=self.xyz[:, 0].round(1), Y_um=self.xyz[:, 1].round(1),\n                                 Z_um=self.xyz[:, 2].round(1), Activity=self.rate.round(3),\n                                 Out_Weight=self.W.sum(axis=1).round(3), In_Weight=self.W.sum(axis=0).round(3)))\n\n    def synapses_table(self):\n        i, j = np.where(self.mask)\n        return pd.DataFrame(dict(Pre_ID=self.nid[i], Post_ID=self.nid[j],\n                                 Pre_Role=[ROLE_NAME[r] for r in self.role[i]],\n                                 Post_Role=[ROLE_NAME[r] for r in self.role[j]],\n                                 Weight=self.W[i, j].round(4), Inhibitory=self.inhib[i]))\n\n    def state(self):\n        d = {k: getattr(self, k).copy() for k in PER_NEURON + PER_PAIR}\n        d.update(next_id=self.next_id, events=list(self.events))\n        return d\n\n    def load(self, d):\n        for k in PER_NEURON + PER_PAIR:\n            if k in d:\n                setattr(self, k, d[k].copy())\n        self.next_id = int(d[\"next_id\"])\n        self.events = list(d[\"events\"])\n", "sim.py": "\"\"\"OIPong: оркестратор. Собирает Network + PongGame + BioEnvironment + правило обучения.\"\"\"\nimport pickle\nimport numpy as np\nimport pandas as pd\nfrom .config import Config, ROLE_SENS, ROLE_HID, ROLE_MOT, cfg_to_dict, cfg_from_dict\nfrom .network import Network\nfrom .game import PongGame\nfrom .environment import BioEnvironment\nfrom .hh import integrate_frame\nfrom .learning import apply_plasticity\n\n\nclass OIPong:\n    def __init__(self, cfg=None):\n        self.cfg = cfg or Config()\n        self.rng = np.random.default_rng(self.cfg.seed)\n        self.net = Network(self.cfg, self.rng)\n        self.game = PongGame(self.cfg)\n        self.env = BioEnvironment(self.cfg)\n        self.episode = 0\n        self.global_frame = 0\n        self.reward_baseline = 0.0\n        self.noise_left = 0\n        self.prev_spk = np.zeros(self.net.N)\n        self.game.reset(self.rng)\n\n    # все свойства сети (W, mask, xyz, pref, idx, mot_up, N, events ...) доступны прямо на OIPong\n    def __getattr__(self, name):\n        if name in (\"net\", \"__setstate__\"):\n            raise AttributeError(name)\n        return getattr(self.net, name)\n\n    # ------------------------------------------------------------------ кадр\n    def _sensory_current(self):\n        c, net = self.cfg, self.net\n        I = np.zeros(net.N)\n        s = net.idx(ROLE_SENS)\n        I[s] = c.i_sens_max * self.game.encode(net.pref)[s]\n        if self.noise_left > 0 and c.feedback_noise:\n            I[s] += self.rng.normal(0, c.i_sens_max * 0.5, len(s))\n            self.noise_left -= 1\n        return I\n\n    def step(self, learn=True):\n        c, net, env = self.cfg, self.net, self.env\n        spikes = integrate_frame(net, c, self._sensory_current(), self.rng, env, explore=learn)\n        up, down = spikes[net.mot_up].sum(), spikes[net.mot_down].sum()\n        move = float(np.clip(c.paddle_gain * (up - down), -c.paddle_max, c.paddle_max))\n        action = \"UP\" if move > 0 else (\"DOWN\" if move < 0 else \"HOLD\")\n\n        paddle_old, done, hit = self.game.advance(move)\n        reward = self.game.shaping_reward(paddle_old)\n        if done:\n            reward += c.r_hit if hit else c.r_miss\n\n        da = reward - (self.reward_baseline if c.use_baseline else 0.0)\n        if learn:\n            if c.use_baseline:\n                self.reward_baseline += 0.02 * (reward - self.reward_baseline)\n            apply_plasticity(net, c, spikes, self.prev_spk, da, lr_gain=env.plasticity_gain)\n            net.C = 0.98 * net.C + 0.02 * np.outer(self.prev_spk > 0, spikes > 0)\n            net.rate = 0.95 * net.rate + 0.05 * (spikes > 0)\n        exc = ~net.inhib\n        env.step(c.frame_ms, n_exc_spikes=float(spikes[exc].sum()), n_inh_spikes=float(spikes[net.inhib].sum()),\n                 reward=reward)\n        self.prev_spk = spikes.copy()\n        self.global_frame += 1\n        g = self.game\n        return dict(spikes=spikes, up=float(up), down=float(down), action=action, move=move, reward=float(reward),\n                    da=float(da), done=done, hit=hit, ball_x=g.ball_x, ball_y=g.ball_y, paddle_y=g.paddle_y,\n                    error=float(g.error), v=net.v.copy())\n\n    def play_episode(self, learn=True, record=False):\n        self.game.reset(self.rng)\n        self.prev_spk = np.zeros(self.net.N)\n        frames, log, t = [], [], 0\n        while True:\n            r = self.step(learn=learn)\n            snap = self.env.snapshot()\n            log.append(dict(Episode=self.episode, Frame=t, Ball_X=round(r[\"ball_x\"], 4), Ball_Y=round(r[\"ball_y\"], 4),\n                            Paddle_Y=round(r[\"paddle_y\"], 4), Error=round(r[\"error\"], 4), Spikes_UP=r[\"up\"],\n                            Spikes_DOWN=r[\"down\"], Action=r[\"action\"], Reward=round(r[\"reward\"], 3),\n                            Dopamine_Signal=round(r[\"da\"], 3), Total_Spikes=float(r[\"spikes\"].sum()),\n                            Hit=r[\"hit\"] if r[\"done\"] else None, Temp_C=snap[\"T\"], Glu=snap[\"Glu\"],\n                            GABA=snap[\"GABA\"], DA_Level=snap[\"DA\"], Plasticity_Gain=snap[\"plasticity_gain\"]))\n            if record:\n                frames.append(dict(ball_x=r[\"ball_x\"], ball_y=r[\"ball_y\"], paddle_y=r[\"paddle_y\"],\n                                   spikes=r[\"spikes\"].copy(), v=r[\"v\"], action=r[\"action\"]))\n            t += 1\n            if r[\"done\"]:\n                break\n        if (not r[\"hit\"]) and learn:\n            self.noise_left = 3\n        return r[\"hit\"], r[\"error\"], frames, log\n\n    def evaluate(self, n=100, seed=777):\n        \"\"\"n эпизодов БЕЗ обучения и БЕЗ шума исследования; фиксированный seed; состояние обучения не меняется.\"\"\"\n        saved = (self.rng, self.noise_left, self.env.state(), self.reward_baseline)\n        self.rng = np.random.default_rng(seed)\n        self.noise_left = 0\n        hits, errs = [], []\n        try:\n            for _ in range(n):\n                hit, err, _, _ = self.play_episode(learn=False)\n                hits.append(bool(hit))\n                errs.append(abs(err))\n        finally:\n            self.rng, self.noise_left, env_state, self.reward_baseline = saved\n            self.env.load(env_state)\n        return dict(win=float(np.mean(hits)), miss_dist=float(np.mean(errs)), n=int(n))\n\n    # --------------------------------------------------------------- обучение\n    def train(self, target=1.0, max_episodes=4000, eval_every=100, eval_n=100, confirm_n=300, window=30,\n              log_every=100, frame_log_every=20, eval_seed=777, verbose=True):\n        \"\"\"Обучаем, пока ОЦЕНКА (без обучения и без шума) не достигнет target; затем подтверждение на другом\n        seed. max_episodes — эпизодов ЭТОГО запуска (None = без лимита). target=None: без ранней остановки.\n        Возвращает (episodes_df, frames_df, eval_df, reached).\"\"\"\n        hist, frame_log, evals, hits = [], [], [], []\n        reached, start = False, self.episode\n\n        def eval_row(extra=None):\n            r = self.evaluate(eval_n, seed=eval_seed)\n            row = dict(Episode=self.episode, Eval_Win_Pct=round(100 * r[\"win\"], 2), Eval_N=r[\"n\"],\n                       Eval_Miss_Distance=round(r[\"miss_dist\"], 4), Neurons_Total=self.net.N,\n                       Synapses=int(self.net.mask.sum()))\n            if extra:\n                row.update(extra)\n            return row\n\n        evals.append(eval_row())\n        if verbose:\n            print(f\"эпизод {self.episode:5d} | eval = {evals[-1]['Eval_Win_Pct']:5.1f}% (до обучения в этом запуске)\")\n        while True:\n            self.episode += 1\n            self.net.episode = self.episode\n            hit, err, _, flog = self.play_episode(learn=True)\n            if frame_log_every and self.episode % frame_log_every == 0:\n                frame_log.extend(flog)\n            hits.append(1 if hit else 0)\n            roll, cum = float(np.mean(hits[-window:])), float(np.mean(hits))\n            hist.append(self._episode_row(hit, err, roll, cum, float(sum(f[\"Reward\"] for f in flog))))\n            if self.cfg.structural and self.episode % self.cfg.struct_every == 0 and len(hits) >= window:\n                self.net.structural_update(roll)\n            if eval_every and self.episode % eval_every == 0:\n                row = eval_row()\n                evals.append(row)\n                if verbose and self.episode % log_every == 0:\n                    print(f\"эпизод {self.episode:5d} | eval = {row['Eval_Win_Pct']:5.1f}% | train-окно({window}) = \"\n                          f\"{roll*100:5.1f}% | нейронов {self.net.N} | синапсов {int(self.net.mask.sum())} | \"\n                          f\"ср.вес {self.net.mean_w():.3f}\")\n                if target is not None and row[\"Eval_Win_Pct\"] >= 100 * target - 1e-9:\n                    conf = self.evaluate(confirm_n, seed=eval_seed + 1)\n                    row[\"Confirm_Win_Pct\"], row[\"Confirm_N\"] = round(100 * conf[\"win\"], 2), conf[\"n\"]\n                    if verbose:\n                        print(f\"  подтверждение на независимом seed: {100*conf['win']:.1f}% ({confirm_n} эп.)\")\n                    if conf[\"win\"] >= target - 1e-9:\n                        reached = True\n                        break\n            if max_episodes is not None and self.episode - start >= max_episodes:\n                break\n        if verbose:\n            tag = \"ЦЕЛЬ ДОСТИГНУТА\" if reached else \"остановлено по лимиту эпизодов\"\n            print(f\"\\n{tag}: всего эпизодов {self.episode}, последняя оценка {evals[-1]['Eval_Win_Pct']:.1f}%\")\n        return pd.DataFrame(hist), pd.DataFrame(frame_log), pd.DataFrame(evals), reached\n\n    def _episode_row(self, hit, err, roll, cum, ep_reward=0.0):\n        net, hid = self.net, self.net.idx(ROLE_HID)\n        snap = self.env.snapshot()\n        return dict(Episode=self.episode, Hit=int(hit), Miss_Distance=round(abs(err), 4),\n                    Episode_Reward=round(ep_reward, 3), WinRate_Rolling_Pct=round(roll * 100, 2),\n                    WinRate_Cumulative_Pct=round(cum * 100, 2), Neurons_Total=net.N, Neurons_Hidden=len(hid),\n                    Synapses=int(net.mask.sum()), Mean_Weight=round(net.mean_w(), 4),\n                    W_Sens_Hid=round(net.class_w(ROLE_SENS, ROLE_HID), 4),\n                    W_Hid_Hid=round(net.class_w(ROLE_HID, ROLE_HID), 4),\n                    W_Hid_Mot=round(net.class_w(ROLE_HID, ROLE_MOT), 4),\n                    W_Sens_Mot=round(net.class_w(ROLE_SENS, ROLE_MOT), 4),\n                    Rate_Sens=round(float(net.rate[net.role == ROLE_SENS].mean()), 3),\n                    Rate_Hid=round(float(net.rate[hid].mean()) if len(hid) else 0.0, 3),\n                    Rate_Mot=round(float(net.rate[net.role == ROLE_MOT].mean()), 3),\n                    Reward_Baseline=round(self.reward_baseline, 4), Temp_C=snap[\"T\"], Glu=snap[\"Glu\"],\n                    GABA=snap[\"GABA\"], DA_Level=snap[\"DA\"], Plasticity_Gain=snap[\"plasticity_gain\"])\n\n    # --------------------------------------------------------- сохранение\n    def state_dict(self):\n        return dict(net=self.net.state(), env=self.env.state(), episode=self.episode,\n                    reward_baseline=self.reward_baseline)\n\n    def load_state(self, d):\n        self.net.load(d[\"net\"])\n        self.env.load(d[\"env\"])\n        self.episode = int(d[\"episode\"])\n        self.net.episode = self.episode\n        self.reward_baseline = float(d[\"reward_baseline\"])\n        self.prev_spk = np.zeros(self.net.N)\n        self.game.reset(self.rng)\n\n\ndef save_game(game, path=\"oi_state.pkl\"):\n    with open(path, \"wb\") as f:\n        pickle.dump(dict(cfg=cfg_to_dict(game.cfg), state=game.state_dict()), f)\n\n\ndef load_game(path=\"oi_state.pkl\"):\n    with open(path, \"rb\") as f:\n        d = pickle.load(f)\n    game = OIPong(cfg_from_dict(d[\"cfg\"]))\n    game.load_state(d[\"state\"])\n    return game\n", "bioconfig.py": "\"\"\"Параметры движка уровня 2 (нейроны Pospischil, HD-MEA, гель, R-STDP). Отдельно от Config v3 (Pong v3 не трогаем).\n\nКаждое значение помечено источником или словом «допущение». Меняйте через make_bio_config(**kw):\nопечатка в имени параметра -> KeyError (как make_config в v3).\n\"\"\"\nimport json\n\n\nclass BioConfig:\n    \"\"\"Все параметры движка уровня 2.\"\"\"\n    seed = 1\n\n    # ---------------- геометрия (мкм). Объём культуры над чипом: MEA в плоскости Z = 0.\n    box = (600.0, 600.0, 150.0)      # допущение: тонкий 3D-слой (органоид/гель) над HD-MEA\n    n_neurons = 1000\n    frac_fs = 0.2                    # 80 % RS / 20 % FS: доля тормозных в коре (учебник, гл. 4; Markram et al. 2004)\n    fs_gM = 0.0                      # FS без M-тока (Pospischil 2008, Fig. 3); 0.0787 = подгонка Fig. 4\n\n    # ---------------- время\n    dt = 0.1                         # мс; экспоненциальный Эйлер. f–I при 0.1 и 0.025 мс расходятся <= 2 % (run_checks fi)\n    syn_delay = 1.0                  # мс, общая задержка проведения (учебник гл. 3)\n\n    # ---------------- синапсы (учебник гл. 3: AMPA 0 мВ / 2 мс, NMDA 0 мВ / 100 мс, ГАМК-А -75 мВ / 6 мс)\n    tau_ampa, tau_nmda, tau_gaba = 2.0, 100.0, 6.0\n    E_gaba = -75.0\n    nmda_ratio = 0.15                # пик g_NMDA / g_AMPA на синапс; допущение (порядок 0.1–0.5 в коре)\n    # Веса W — прирост проводимости на спайк, мСм/см2. Для RS (g_L = 0.0205 мСм/см2) прирост 0.004\n    # даёт ВПСП ~0.5 мВ у покоя (оценка: dV ~ dg * tau * (E - V) / C). Значения подобраны (допущение).\n    w_ee, w_ei, w_ie, w_ii = 0.0045, 0.006, 0.012, 0.015\n    w_max_ee = 0.04                  # потолок пластичных E->E весов (оракул: при 0.016 потолок точности ~80 %)\n    # связность: p = p0[класс] * exp(-d / lambda) при d <= conn_radius (радиус режет гель, gel.py)\n    p0_ee, p0_ei, p0_ie, p0_ii = 0.25, 0.35, 0.55, 0.45     # допущение\n    conn_lambda = 120.0              # мкм; допущение\n    conn_radius_ref = 160.0          # мкм при опорной вязкости геля (gel.py)\n    n_long = 20                      # дальних аксонов на RS (учебник гл. 4: 20 на нейрон; без них дальняя треть\n                                     # на уровне шанса). Замер: при 10 seed 2 учился хуже (README)\n    long_spread = 250.0              # мкм, масштаб длины дальних связей; допущение\n\n    # ---------------- фон (синаптический шум коры), учебник гл. 4: без фона сеть молчит\n    # Редкие крупные события (200 Гц x 0.016) дают нерегулярную активность; при 600 Гц x 0.008 нейроны\n    # стреляли почти периодически, разброс ответов мал и R-STDP не учился (замер: README).\n    bg_rate = 200.0                  # Гц пуассоновских внешних AMPA-событий на нейрон; допущение\n    bg_w = 0.016                     # мСм/см2 на событие; допущение\n    bg_w_fs_mult = 0.9               # множитель фона для FS; подобран (допущение)\n\n    # ---------------- гелевая среда (gel.py)\n    temperature = 36.0               # °C\n    temp_ref = 36.0                  # кинетика Pospischil 2008 дана при 36 °C\n    q10_gate = 3.0                   # Q10 скоростей затворов: ~3 (Hodgkin & Huxley 1952; Hille 2001)\n    q10_cond = 1.3                   # Q10 максимальных проводимостей ~1.2–1.5 (Hille 2001); допущение в выборе 1.3\n    q10_syn = 2.0                    # Q10 скорости спада синаптических токов; допущение (порядок 2–3)\n    mg_mM = 1.0                      # [Mg2+]o, мМ (ТЗ); блок NMDA по Jahr & Stevens 1990\n    viscosity = 1.0                  # отн. вязкость геля (1.0 = опорная)\n    visc_exponent = 0.5              # conn_radius = ref * (eta_ref/eta)^0.5; допущение (качественно: рост\n                                     # нейритов медленнее в более вязкой/жёсткой матрице, Balgude et al. 2001)\n    # внеклеточный K+ (диффузия на сетке вокселей)\n    k_enabled = True\n    k_rest = 3.0                     # мМ, [K+]o покоя; допущение (ИСЖ 2.5–3.5 мМ)\n    k_voxel = 50.0                   # мкм, шаг сетки\n    k_D = 1.96                       # мкм2/мс = 1.96e-5 см2/с, K+ в воде (CRC Handbook)\n    k_tortuosity = 1.6               # Nicholson & Syková 1998 (мозг); для геля меньше — допущение\n    k_alpha = 0.2                    # доля внеклеточного объёма (Nicholson & Syková 1998)\n    k_per_spike_fmol = 0.36          # фмоль K+ на спайк (≈3 x C*dV*A/F для RS); допущение\n    k_tau_clear = 1000.0             # мс, захват глией/насосами; допущение\n    k_update_ms = 1.0\n\n    # ---------------- HD-MEA (mea.py)\n    mea_pitch = 30.0                 # мкм; реальные HD-MEA 17.5 мкм (MaxWell), берём реже ради скорости\n    mea_r0 = 35.0                    # мкм, масштаб затухания сигнала: a = 1/(1 + (r/r0)^2); допущение\n    mea_noise = 0.05                 # шум записи, отн. ед. на бин\n    stim_sigma = 60.0                # мкм, ширина гауссова поля деполяризации\n    stim_amp = 5.0                   # мкА/см2 в центре поля (ось электрода, Z = 0)\n    stim_jitter_amp = 0.2            # разброс амплитуды от пробы к пробе (±20 %)\n    stim_jitter_pos = 15.0           # мкм, разброс центра\n\n    # ---------------- задача A/B (task_ab.py)\n    # Геометрия задачи симметрична: каждый выходной кластер РАВНОУДАЛЁН от A и от B (212 мкм),\n    # поэтому расположение электродов не подсказывает ответ: необученная сеть ~50 %.\n    stim_A = (150.0, 300.0)          # центр стимулирующего кластера A (x, y), мкм\n    stim_B = (450.0, 300.0)\n    out_1 = (300.0, 150.0)           # «моторные» электроды: A -> out_1, B -> out_2\n    out_2 = (300.0, 450.0)\n    center_decision = True           # решение по (сигнал - его скользящее среднее), см. task_ab.py\n    center_rate = 0.05               # скорость скользящего среднего (по пробам)\n    cluster_radius = 50.0            # мкм: электроды кластера в этом радиусе от центра\n    mapping = \"straight\"             # 'straight': A -> out_1, B -> out_2 | 'crossed': наоборот\n    t_pre, t_stim, t_post = 40.0, 60.0, 110.0   # мс: пауза, стимул (он же окно решения), пауза после награды\n    decision_lag = 10.0              # мс: окно решения = [начало стимула, конец стимула + lag]\n\n    # ---------------- обучение: R-STDP + след пригодности + дофамин + синаптическое масштабирование\n    tau_plus, tau_minus = 20.0, 20.0     # мс, окно STDP (Bi & Poo 1998; Song, Miller & Abbott 2000)\n    A_plus = 1.0\n    A_minus = 1.05                       # A-/A+ = 1.05 (Song et al. 2000)\n    tau_elig = 200.0                     # мс, след пригодности (Izhikevich 2007: 1 с; у нас награда через\n                                         # 70 мс после начала стимула, короткий след меньше шумит; подобрано)\n    lr = 0.001                           # мСм/см2 на единицу (дофамин * след); подобрано\n    lr_decay_trials = 900                # lr / (1 + проба / 900): к 1800-й пробе lr втрое меньше — закрепление; 0 = выкл.\n    reward_mode = \"binary\"               # 'binary': R = +1/-1 | 'margin': R = tanh(запас решения / его СКО)\n    critic_rate = 0.1                    # обучение ожидаемой награды по стимулу (Frémaux et al. 2010)\n    da_gain = 1.0                        # множитель фазического дофамина (тонический DA из BioEnvironment)\n    scaling = True                       # синаптическое масштабирование (Turrigiano et al. 1998)\n    rate_target_rs, rate_target_fs = 3.5, 10.0   # Гц (учебник гл. 4: ~3.6 / ~11 Гц в рабочем режиме)\n    scale_rate = 0.02                    # шаг масштабирования за пробу\n    scale_tau_trials = 10.0              # усреднение частоты по пробам\n    scale_clip = 0.03                    # |изменение| за пробу <= 3 %\n\n\ndef make_bio_config(**kw):\n    c = BioConfig()\n    for k, v in kw.items():\n        if not hasattr(c, k):\n            raise KeyError(f\"Неизвестный параметр BioConfig: {k}\")\n        setattr(c, k, v)\n    return c\n\n\ndef bio_cfg_to_dict(cfg):\n    out = {}\n    for k in dir(BioConfig):\n        if k.startswith(\"_\") or callable(getattr(BioConfig, k)):\n            continue\n        v = getattr(cfg, k)\n        out[k] = list(v) if isinstance(v, tuple) else v\n    return out\n\n\ndef save_bio_config(cfg, path=\"bio_cfg.json\"):\n    json.dump(bio_cfg_to_dict(cfg), open(path, \"w\"), ensure_ascii=False, indent=1)\n", "neuron.py": "\"\"\"Нейрон уровня 2: проводимостная модель Pospischil et al. (2008).\n\nИсточник: Pospischil M., Toledo-Rodriguez M., Monier C., Piwkowska Z., Bal T., Frégnac Y., Markram H.,\nDestexhe A. «Minimal Hodgkin–Huxley type models for different classes of cortical and thalamic neurons».\nBiol. Cybern. 99:427–441 (2008). Кинетика каналов в статье дана для температуры 36 °C.\n\nОдно-компартментный нейрон (цилиндр L = d), уравнение мембраны (ур. 1 статьи):\n\n    C_m dV/dt = - g_L (V - E_L) - I_Na - I_Kd - I_M - I_syn + I_ext\n\n    I_Na = g_Na m^3 h (V - E_Na)        быстрый натрий, Traub & Miles 1991 (ур. 4)\n    I_Kd = g_Kd n^4   (V - E_K)         задержанный выпрямитель калия (ур. 5)\n    I_M  = g_M  p     (V - E_K)         медленный K+ (M-ток), Yamada et al. 1989: адаптация частоты у RS\n\nСкорости (1/мс), V в мВ, V_T сдвигает порог спайка:\n    a_m = -0.32 (V-VT-13) / (exp(-(V-VT-13)/4) - 1)      b_m = 0.28 (V-VT-40) / (exp((V-VT-40)/5) - 1)\n    a_h = 0.128 exp(-(V-VT-17)/18)                        b_h = 4 / (1 + exp(-(V-VT-40)/5))\n    a_n = -0.032 (V-VT-15) / (exp(-(V-VT-15)/5) - 1)      b_n = 0.5 exp(-(V-VT-10)/40)\n    p_inf = 1 / (1 + exp(-(V+35)/10))\n    tau_p = tau_max / (3.3 exp((V+35)/20) + exp(-(V+35)/20))\n\nЕдиницы: C_m = 1 мкФ/см2, проводимости в мСм/см2, токи в мкА/см2 (=> dV/dt в мВ/мс), время в мс.\nПересчёт в абсолютные единицы: площадь цилиндра L = d равна pi*d*L (торцы в статье не учитываются):\nRS d = 61.4 мкм -> 1.18e-4 см2, 1 мкА/см2 = 0.118 нА.\n\nПараметры клеток (взяты из подписей к рисункам статьи, сверены с PDF):\n  RS (возбуждающая пирамидная, Fig. 2a, соматосенсорная кора крысы): L = d = 61.4 мкм,\n      g_L = 2.05e-5 См/см2 (0.0205 мСм/см2), E_L = -70.3 мВ, g_Na = 0.056 См/см2, V_T = -56.2 мВ,\n      g_Kd = 0.006 См/см2, g_M = 7.5e-5 См/см2, tau_max = 608 мс.\n  FS (быстрая тормозная, Fig. 4, та же кора): L = d = 56.9 мкм, g_L = 3.8e-5 См/см2, E_L = -70.4 мВ,\n      g_Na = 0.058 См/см2, V_T = -57.9 мВ, g_Kd = 0.0039 См/см2.\n      В Fig. 4 у подогнанной FS-клетки есть слабый M-ток (7.87e-5 См/см2, tau_max 502 мс); по ТЗ и по модели\n      FS из Fig. 3 статьи («no I_M») мы его ОБНУЛЯЕМ: FS без адаптации. Включить: fs_gM=0.0787.\n  E_Na = 50 мВ, E_K = -90 мВ (раздел 2.2 статьи).\n\nИнтегрирование: экспоненциальный метод Эйлера (Rush–Larsen для затворов и для V при замороженных\nпроводимостях): x(t+dt) = x_inf + (x - x_inf) * exp(-dt / tau_x). Он безусловно устойчив для линейного по x\nуравнения; шаг 0.05 мс (точность проверяется сравнением с 0.025 мс в run_checks.py).\n\nТемпература: скорости затворов умножаются на phi = Q10_gate ** ((T - 36) / 10) (см. gel.py).\n\"\"\"\nimport numpy as np\n\nE_NA, E_K = 50.0, -90.0      # мВ, Pospischil 2008, раздел 2.2\nT_REF_KIN = 36.0             # °C, температура, к которой относится кинетика (Pospischil 2008, раздел 2.2)\n\n# Параметры типов (мСм/см2, мВ, мс, мкм). Индекс типа: 0 = RS, 1 = FS.\nRS, FS = 0, 1\nTYPE_NAME = {RS: \"RS\", FS: \"FS\"}\nCELL_PARAMS = {\n    RS: dict(L=61.4, gL=0.0205, EL=-70.3, gNa=56.0, VT=-56.2, gKd=6.0, gM=0.075, tau_max=608.0),\n    FS: dict(L=56.9, gL=0.038, EL=-70.4, gNa=58.0, VT=-57.9, gKd=3.9, gM=0.0, tau_max=502.0),\n}\n\n\ndef area_cm2(L_um):\n    \"\"\"Площадь боковой поверхности цилиндра L = d, см2 (1 мкм2 = 1e-8 см2).\"\"\"\n    return np.pi * L_um * L_um * 1e-8\n\n\ndef _vtrap(x, y):\n    \"\"\"x / (exp(x/y) - 1) без деления 0/0 при x -> 0 (предел равен y).\"\"\"\n    x = np.asarray(x, dtype=float)\n    small = np.abs(x / y) < 1e-6\n    xs = np.where(small, 1.0, x)\n    return np.where(small, y * (1.0 - x / y / 2.0), xs / np.expm1(xs / y))\n\n\ndef rates_u(u):\n    \"\"\"Скорости Na и Kd как функция u = V - V_T (мВ). Возвращает (a_m, b_m, a_h, b_h, a_n, b_n), 1/мс.\"\"\"\n    a_m = 0.32 * _vtrap(-(u - 13.0), 4.0)          # = -0.32 (u-13) / (exp(-(u-13)/4) - 1)\n    b_m = 0.28 * _vtrap(u - 40.0, 5.0)             # =  0.28 (u-40) / (exp((u-40)/5) - 1)\n    a_h = 0.128 * np.exp(-(u - 17.0) / 18.0)\n    b_h = 4.0 / (1.0 + np.exp(-(u - 40.0) / 5.0))\n    a_n = 0.032 * _vtrap(-(u - 15.0), 5.0)\n    b_n = 0.5 * np.exp(-(u - 10.0) / 40.0)\n    return a_m, b_m, a_h, b_h, a_n, b_n\n\n\ndef m_current_kin(V, tau_max):\n    \"\"\"M-ток: p_inf(V) и tau_p(V) (мс), Yamada et al. 1989 в редакции Pospischil 2008.\"\"\"\n    p_inf = 1.0 / (1.0 + np.exp(-(V + 35.0) / 10.0))\n    tau_p = tau_max / (3.3 * np.exp((V + 35.0) / 20.0) + np.exp(-(V + 35.0) / 20.0))\n    return p_inf, tau_p\n\n\nclass GateTables:\n    \"\"\"Таблицы x_inf и exp(-dt*phi/tau) на сетке напряжения (шаг 0.02 мВ): ускорение в ~5 раз.\n    Na/Kd табулируются по u = V - V_T (общая таблица для RS и FS), M-ток по V для каждого tau_max.\"\"\"\n    LO, HI, RES = -160.0, 120.0, 50.0       # мВ, точек на мВ\n\n    def __init__(self, dt, phi, tau_maxes):\n        self.dt, self.phi = dt, phi\n        g = np.arange(self.LO, self.HI + 1.0 / self.RES, 1.0 / self.RES)\n        self.grid = g\n        a_m, b_m, a_h, b_h, a_n, b_n = rates_u(g)\n        k = dt * phi\n        self.m_inf, self.m_e = a_m / (a_m + b_m), np.exp(-k * (a_m + b_m))\n        self.h_inf, self.h_e = a_h / (a_h + b_h), np.exp(-k * (a_h + b_h))\n        self.n_inf, self.n_e = a_n / (a_n + b_n), np.exp(-k * (a_n + b_n))\n        self.p = {}\n        for tm in set(tau_maxes):\n            p_inf, tau_p = m_current_kin(g, tm)\n            self.p[tm] = (p_inf, np.exp(-k / tau_p))\n        self.top = len(g) - 1\n\n    def index(self, x):\n        return np.clip(((x - self.LO) * self.RES + 0.5).astype(np.int64), 0, self.top)\n\n\ndef steady_state(V, VT, tau_max=608.0):\n    \"\"\"Стационарные значения затворов при напряжении V: (m, h, n, p).\"\"\"\n    a_m, b_m, a_h, b_h, a_n, b_n = rates_u(np.asarray(V) - VT)\n    p_inf, _ = m_current_kin(np.asarray(V), tau_max)\n    return a_m / (a_m + b_m), a_h / (a_h + b_h), a_n / (a_n + b_n), p_inf\n\n\nclass NeuronArray:\n    \"\"\"Массив одно-компартментных нейронов Pospischil (векторно по numpy).\n    Состояние: V, m, h, n, p. Параметры — по типу (RS/FS) в массивах длины N.\"\"\"\n\n    def __init__(self, types, dt=0.05, phi=1.0, cond_factor=1.0, fs_gM=0.0, V0=None):\n        self.types = np.asarray(types, dtype=int)\n        N = len(self.types)\n        self.N, self.dt = N, dt\n        P = {k: np.array([CELL_PARAMS[t][k] for t in self.types], dtype=float) for k in CELL_PARAMS[RS]}\n        if fs_gM:\n            P[\"gM\"][self.types == FS] = fs_gM\n        self.P = P\n        self.area = area_cm2(P[\"L\"])\n        self.set_temperature_factors(phi, cond_factor)\n        V0 = np.where(self.types == RS, CELL_PARAMS[RS][\"EL\"], CELL_PARAMS[FS][\"EL\"]) if V0 is None else V0\n        self.V = np.array(V0, dtype=float) * np.ones(N)\n        m, h, n, p = steady_state(self.V, P[\"VT\"], 608.0)\n        self.m, self.h, self.n = m.copy(), h.copy(), n.copy()\n        self.p = np.array([m_current_kin(v, tm)[0] for v, tm in zip(self.V, P[\"tau_max\"])])\n        self.above = np.zeros(N, dtype=bool)\n        self.EK_shift = np.zeros(N)      # сдвиг E_K от внеклеточного K+ (gel.py), мВ\n\n    def set_temperature_factors(self, phi, cond_factor=1.0):\n        \"\"\"phi: множитель скоростей затворов (Q10), cond_factor: множитель максимальных проводимостей.\"\"\"\n        self.phi, self.cond_factor = phi, cond_factor\n        self.tab = GateTables(self.dt, phi, self.P[\"tau_max\"])\n        self._tm_groups = [(tm, np.where(self.P[\"tau_max\"] == tm)[0]) for tm in sorted(set(self.P[\"tau_max\"]))]\n        c = cond_factor\n        self.gNa, self.gKd, self.gM, self.gL = c * self.P[\"gNa\"], c * self.P[\"gKd\"], c * self.P[\"gM\"], c * self.P[\"gL\"]\n\n    def step(self, I_ext, g_exc=None, g_nmda_eff=None, g_inh=None, E_inh=-75.0):\n        \"\"\"Один шаг dt. I_ext: мкА/см2 (вход положительный = деполяризация).\n        g_exc (AMPA, E=0), g_nmda_eff (NMDA с уже умноженным блоком Mg, E=0), g_inh (ГАМК-А, E=E_inh): мСм/см2.\n        Возвращает булев массив «спайк на этом шаге» (пересечение 0 мВ снизу вверх).\"\"\"\n        tab, V = self.tab, self.V\n        iu = tab.index(V - self.P[\"VT\"])\n        self.m = tab.m_inf[iu] + (self.m - tab.m_inf[iu]) * tab.m_e[iu]\n        self.h = tab.h_inf[iu] + (self.h - tab.h_inf[iu]) * tab.h_e[iu]\n        self.n = tab.n_inf[iu] + (self.n - tab.n_inf[iu]) * tab.n_e[iu]\n        iv = tab.index(V)\n        for tm, idx in self._tm_groups:\n            p_inf, p_e = tab.p[tm]\n            j = iv[idx]\n            self.p[idx] = p_inf[j] + (self.p[idx] - p_inf[j]) * p_e[j]\n        EK = E_K + self.EK_shift\n        gna = self.gNa * self.m ** 3 * self.h\n        gk = self.gKd * self.n ** 4 + self.gM * self.p\n        G = gna + gk + self.gL                      # полная проводимость, мСм/см2\n        GE = gna * E_NA + gk * EK + self.gL * self.P[\"EL\"]\n        if g_exc is not None:\n            G = G + g_exc                           # E_AMPA = 0 -> вклад в GE нулевой\n        if g_nmda_eff is not None:\n            G = G + g_nmda_eff\n        if g_inh is not None:\n            G = G + g_inh\n            GE = GE + g_inh * E_inh\n        V_inf = (GE + I_ext) / G                    # мВ; C_m = 1 мкФ/см2 -> tau_V = 1/G мс\n        self.V = V_inf + (V - V_inf) * np.exp(-self.dt * G)\n        now = self.V > 0.0\n        sp = now & ~self.above\n        self.above = now\n        return sp\n\n\ndef simulate_single(cell_type, I_uA_cm2, t_ms=1000.0, dt=0.05, phi=1.0, delay_ms=100.0, record=False, fs_gM=0.0):\n    \"\"\"Одиночный нейрон, ступенька тока I (мкА/см2) с delay_ms до t_ms. Возвращает времена спайков (мс)\n    и, если record, словарь трасс V, m, h, n, p (каждый 4-й шаг).\"\"\"\n    nrn = NeuronArray([cell_type], dt=dt, phi=phi, fs_gM=fs_gM)\n    steps = int(round(t_ms / dt))\n    spikes, rec = [], {k: [] for k in (\"t\", \"V\", \"m\", \"h\", \"n\", \"p\")}\n    for s in range(steps):\n        t = s * dt\n        I = I_uA_cm2 if t >= delay_ms else 0.0\n        if nrn.step(np.array([I]))[0]:\n            spikes.append(t)\n        if record and s % 4 == 0:\n            rec[\"t\"].append(t)\n            for k in (\"V\", \"m\", \"h\", \"n\", \"p\"):\n                rec[k].append(float(getattr(nrn, k)[0]))\n    return np.array(spikes), ({k: np.array(v) for k, v in rec.items()} if record else None)\n\n\ndef fi_curve(cell_type, currents, t_ms=1100.0, delay_ms=100.0, dt=0.05, phi=1.0):\n    \"\"\"f–I кривая: для каждого тока — средняя частота за 1 с стимула, частота первых двух спайков\n    (1/ISI1) и последних (1/ISI_last) и индекс адаптации ISI_last/ISI_first.\"\"\"\n    rows = []\n    for I in currents:\n        sp, _ = simulate_single(cell_type, I, t_ms=t_ms, dt=dt, phi=phi, delay_ms=delay_ms)\n        sp = sp[sp >= delay_ms]\n        isi = np.diff(sp)\n        rows.append(dict(I_uA_cm2=float(I), I_nA=round(float(I * area_cm2(CELL_PARAMS[cell_type][\"L\"]) * 1e3), 4),\n                         rate_Hz=len(sp) / ((t_ms - delay_ms) / 1000.0),\n                         f_first_Hz=1000.0 / isi[0] if len(isi) else 0.0,\n                         f_last_Hz=1000.0 / isi[-1] if len(isi) else 0.0,\n                         adapt=float(isi[-1] / isi[0]) if len(isi) > 1 else np.nan))\n    return rows\n", "gel.py": "\"\"\"Гелевая среда биочипа: температура (Q10), магний и блок NMDA, вязкость -> радиус связности,\nдиффузия внеклеточного K+ на сетке вокселей.\n\n1. Температура (Q10). Скорость процесса при T: k(T) = k(T_ref) * Q10 ** ((T - T_ref) / 10).\n   - затворы m, h, n, p: Q10_gate = 3 (Hodgkin & Huxley 1952; Hille 2001: 2–4 для воротных процессов);\n   - максимальные проводимости: Q10_cond ~ 1.3 (Hille 2001: проводимость одиночного канала 1.2–1.5);\n     обоснование: проводимость ограничена диффузией ионов в поре, её Q10 близок к Q10 диффузии (~1.3);\n   - спад синаптических проводимостей: Q10_syn = 2 (допущение).\n   T_ref = 36 °C: при ней даны кинетики Pospischil et al. 2008.\n\n2. Блок NMDA ионом Mg2+ (Jahr & Stevens 1990, J. Neurosci. 10:3178):\n       B(V) = 1 / (1 + [Mg2+]o / 3.57 мМ * exp(-0.062 V))         V в мВ\n   При [Mg] = 1 мМ и V = -70 мВ B ~ 0.04 (канал почти закрыт), при V = 0 мВ B ~ 0.78.\n\n3. Вязкость геля -> предел радиуса связности (ДОПУЩЕНИЕ, феноменология):\n       conn_radius = conn_radius_ref * (eta_ref / eta) ** visc_exponent\n   Качественная опора: в более плотной/жёсткой 3D-матрице нейриты растут медленнее и короче\n   (Balgude et al. 2001, Biomaterials 22:1077, агарозные гели). Количественной формулы в литературе мы не нашли.\n\n4. Внеклеточный K+: каждый спайк выбрасывает k_per_spike_fmol K+ в воксель нейрона;\n       d[K]/dt = D* лапласиан([K]) - ([K] - K_rest) / tau_clear + выброс / (alpha * V_voxel)\n   D* = D / lambda^2 (извилистость, Nicholson & Syková 1998). Граничное условие: нулевой поток.\n   Сдвиг потенциала реверсии калия по Нернсту:  E_K = E_K0 + (RT/F) ln([K]o / K_rest),\n   RT/F = 26.7 мВ при 36 °C. Утечка не сдвигается (упрощение).\n\"\"\"\nimport numpy as np\n\nR_GAS, F_FARADAY = 8.314, 96485.0\n\n\ndef q10_factor(q10, T, T_ref):\n    \"\"\"Q10 ** ((T - T_ref) / 10).\"\"\"\n    return float(q10 ** ((T - T_ref) / 10.0))\n\n\ndef mg_block(V, mg_mM=1.0):\n    \"\"\"Доля незаблокированных NMDA-каналов, Jahr & Stevens 1990.\"\"\"\n    return 1.0 / (1.0 + (mg_mM / 3.57) * np.exp(-0.062 * V))\n\n\ndef conn_radius_from_viscosity(cfg):\n    \"\"\"Радиус связности (мкм) при относительной вязкости cfg.viscosity. Допущение, см. модуль.\"\"\"\n    return float(cfg.conn_radius_ref * (1.0 / max(cfg.viscosity, 1e-6)) ** cfg.visc_exponent)\n\n\nclass GelMedium:\n    \"\"\"Состояние среды: температура, магний, поле [K+]o.\"\"\"\n\n    def __init__(self, cfg, xyz=None):\n        self.cfg = cfg\n        self.T = float(cfg.temperature)\n        self.mg = float(cfg.mg_mM)\n        self.conn_radius = conn_radius_from_viscosity(cfg)\n        self._init_k(xyz)\n\n    # -------- температура\n    @property\n    def phi_gate(self):\n        return q10_factor(self.cfg.q10_gate, self.T, self.cfg.temp_ref)\n\n    @property\n    def phi_cond(self):\n        return q10_factor(self.cfg.q10_cond, self.T, self.cfg.temp_ref)\n\n    @property\n    def phi_syn(self):\n        return q10_factor(self.cfg.q10_syn, self.T, self.cfg.temp_ref)\n\n    @property\n    def rt_f(self):\n        \"\"\"RT/F в мВ при текущей температуре.\"\"\"\n        return 1000.0 * R_GAS * (self.T + 273.15) / F_FARADAY\n\n    def nmda_block(self, V):\n        return mg_block(V, self.mg)\n\n    # -------- калий\n    def _init_k(self, xyz):\n        c = self.cfg\n        self.k_shape = tuple(int(np.ceil(b / c.k_voxel)) for b in c.box)\n        self.K = np.full(self.k_shape, c.k_rest)\n        self.vox = None\n        if xyz is not None:\n            self.vox = np.clip((xyz / c.k_voxel).astype(int), 0, np.array(self.k_shape) - 1)\n            self.vox_flat = np.ravel_multi_index(self.vox.T, self.k_shape)\n        # концентрация от одного спайка в одном вокселе, мМ: фмоль / (alpha * V_voxel)\n        vol_L = (c.k_voxel * 1e-6) ** 3 * 1e3            # мкм -> м, м3 -> л\n        self.k_per_spike_mM = c.k_per_spike_fmol * 1e-15 / (c.k_alpha * vol_L) * 1e3\n        self.D_eff = c.k_D / c.k_tortuosity ** 2         # мкм2/мс\n\n    def k_step(self, spike_counts, dt_ms):\n        \"\"\"Шаг диффузии K+ за dt_ms (явная схема; устойчива при D*dt/h^2 < 1/6, у нас ~1e-4).\"\"\"\n        c = self.cfg\n        if not c.k_enabled or self.vox is None:\n            return\n        if spike_counts is not None and spike_counts.any():\n            add = np.bincount(self.vox_flat, weights=spike_counts, minlength=self.K.size)\n            self.K += add.reshape(self.k_shape) * self.k_per_spike_mM\n        K = self.K\n        P = np.pad(K, 1, mode=\"edge\")                     # нулевой поток на границах\n        lap = (P[2:, 1:-1, 1:-1] + P[:-2, 1:-1, 1:-1] + P[1:-1, 2:, 1:-1] + P[1:-1, :-2, 1:-1] +\n               P[1:-1, 1:-1, 2:] + P[1:-1, 1:-1, :-2] - 6.0 * K) / c.k_voxel ** 2\n        self.K = K + dt_ms * (self.D_eff * lap - (K - c.k_rest) / c.k_tau_clear)\n\n    def ek_shift(self):\n        \"\"\"Сдвиг E_K (мВ) для каждого нейрона: (RT/F) ln([K]o / K_rest).\"\"\"\n        if self.vox is None or not self.cfg.k_enabled:\n            return 0.0\n        k_here = self.K.ravel()[self.vox_flat]\n        return self.rt_f * np.log(np.maximum(k_here, 1e-3) / self.cfg.k_rest)\n\n    def snapshot(self):\n        return dict(T=self.T, phi_gate=round(self.phi_gate, 3), phi_cond=round(self.phi_cond, 3), Mg_mM=self.mg,\n                    conn_radius=round(self.conn_radius, 1), K_mean=round(float(self.K.mean()), 4),\n                    K_max=round(float(self.K.max()), 4))\n", "mea.py": "\"\"\"HD-MEA: матрица электродов в плоскости Z = 0 под культурой.\n\nСчитывание. Электрод e видит спайк нейрона i с весом, затухающим с расстоянием r_ei (в 3D, поэтому\nглубина Z нейрона входит автоматически):\n        a_ei = 1 / (1 + (r_ei / r0)^2)\nЭто ДОПУЩЕНИЕ: точечный источник тока в однородной среде даёт 1/r (Nunez & Srinivasan 2006), а реальный\nспайк — диполь, его амплитуда падает круче (~1/r^2 и быстрее); спайки на практике видны в пределах\n~100 мкм от электрода (Buzsáki et al. 2012, Nat. Rev. Neurosci. 13:407). Форма 1/(1+(r/r0)^2) с r0 = 35 мкм\nдаёт 1 у самого электрода, 0.5 на 35 мкм и ~0.11 на 100 мкм. Сигнал электрода в бин = сумма a_ei * спайки_i + шум.\n\nСтимуляция. Электрод с амплитудой I0 создаёт у нейрона i деполяризующий ток (ТЗ, гауссово поле):\n        I_i = I0 * exp(-r_ei^2 / (2 sigma^2))         мкА/см2\nРеальная внеклеточная стимуляция действует через «активирующую функцию» (вторую производную потенциала\nвдоль аксона, Rattay 1989); гауссова деполяризация — её грубая замена (допущение ТЗ).\n\"\"\"\nimport numpy as np\n\n\nclass HDMEA:\n    def __init__(self, cfg, xyz):\n        self.cfg = cfg\n        bx, by = cfg.box[0], cfg.box[1]\n        p = cfg.mea_pitch\n        xs = np.arange(p / 2, bx, p)\n        ys = np.arange(p / 2, by, p)\n        self.nx, self.ny = len(xs), len(ys)\n        gx, gy = np.meshgrid(xs, ys, indexing=\"ij\")\n        self.exy = np.stack([gx.ravel(), gy.ravel()], axis=1)           # (E, 2), Z = 0\n        self.n_el = len(self.exy)\n        self.set_neurons(xyz)\n\n    def set_neurons(self, xyz):\n        self.xyz = xyz\n        d2 = ((self.exy[:, None, 0] - xyz[None, :, 0]) ** 2 + (self.exy[:, None, 1] - xyz[None, :, 1]) ** 2\n              + xyz[None, :, 2] ** 2)\n        self.r2 = d2                                                     # (E, N), мкм2\n        self.A = 1.0 / (1.0 + d2 / self.cfg.mea_r0 ** 2)                 # матрица считывания\n\n    def electrodes_near(self, center, radius):\n        \"\"\"Индексы электродов в радиусе radius от точки center (x, y).\"\"\"\n        d = np.hypot(self.exy[:, 0] - center[0], self.exy[:, 1] - center[1])\n        idx = np.where(d <= radius)[0]\n        return idx if len(idx) else np.array([int(np.argmin(d))])\n\n    def read(self, spikes, rng=None):\n        \"\"\"Сигнал всех электродов для вектора спайков (или числа спайков за бин) нейронов.\"\"\"\n        s = self.A @ spikes\n        if rng is not None and self.cfg.mea_noise > 0:\n            s = s + rng.normal(0.0, self.cfg.mea_noise, s.shape)\n        return s\n\n    def stim_current(self, centers_xy, amp, sigma=None):\n        \"\"\"Ток деполяризации (мкА/см2) каждого нейрона от набора точек стимуляции (x, y) на Z = 0.\"\"\"\n        sigma = self.cfg.stim_sigma if sigma is None else sigma\n        I = np.zeros(len(self.xyz))\n        for cx, cy in np.atleast_2d(centers_xy):\n            r2 = (self.xyz[:, 0] - cx) ** 2 + (self.xyz[:, 1] - cy) ** 2 + self.xyz[:, 2] ** 2\n            I += amp * np.exp(-r2 / (2.0 * sigma ** 2))\n        return I\n\n    def stim_from_electrodes(self, el_idx, amp, sigma=None):\n        return self.stim_current(self.exy[el_idx], amp, sigma)\n\n    def to_image(self, sig):\n        \"\"\"Вектор сигналов электродов -> 2D-массив (nx, ny) для тепловой карты.\"\"\"\n        return np.asarray(sig).reshape(self.nx, self.ny)\n", "bionet.py": "\"\"\"BioNet: 3D-сеть нейронов Pospischil (RS/FS) над HD-MEA в гелевой среде.\n\nСборка: позиции в объёме box, типы RS/FS, связность по расстоянию (радиус режет гель), дальние аксоны,\nсинапсы-проводимости AMPA/NMDA/ГАМК-А с задержкой 1 мс, пуассоновский фон, поле K+.\nСимуляция идёт кусками (run), между кусками задача (task_ab.py) меняет стимул и выдаёт награду.\n\nСинаптические токи (мкА/см2), суммарные проводимости на постсинаптическом нейроне:\n    I_syn = g_AMPA (V - 0) + g_NMDA B(V) (V - 0) + g_GABA (V - E_GABA)\n    dg/dt = -g / tau   (+ W_ij при приходе спайка от i через syn_delay)\nB(V) — блок Mg2+ (Jahr & Stevens 1990, gel.py). Тип синапса задаёт пресинаптический нейрон:\nRS -> AMPA + NMDA (nmda_ratio * W), FS -> ГАМК-А (закон Дейла).\n\"\"\"\nimport numpy as np\nfrom .neuron import NeuronArray, RS, FS\nfrom .gel import GelMedium\nfrom .mea import HDMEA\n\n\nclass Recorder:\n    \"\"\"Что записывать во время run(). Всё опционально, чтобы обучение шло быстро.\"\"\"\n\n    def __init__(self, spikes=True, bin_ms=5.0, v_every_ms=None, gate_ids=(), gate_every=4, mea=True, k_every_ms=None,\n                 balance=True):\n        self.want_spikes, self.bin_ms, self.v_every_ms = spikes, bin_ms, v_every_ms\n        self.gate_ids, self.gate_every, self.want_mea = list(gate_ids), gate_every, mea\n        self.k_every_ms, self.want_balance = k_every_ms, balance\n        self.reset()\n\n    def reset(self):\n        self.spk_t, self.spk_i = [], []\n        self.bins_t, self.bin_counts, self.mea_bins = [], [], []\n        self.v_t, self.v_snap = [], []\n        self.gate_t, self.gates = [], {k: [] for k in (\"V\", \"m\", \"h\", \"n\", \"p\")}\n        self.k_t, self.k_snap = [], []\n        self.bal_t, self.bal_exc, self.bal_inh = [], [], []\n        self.stim_label = []          # (t0, t1, label) — отметки стимулов для графиков\n\n    def arrays(self):\n        out = dict(spk_t=np.array(self.spk_t), spk_i=np.array(self.spk_i, dtype=int),\n                   bins_t=np.array(self.bins_t), bin_counts=np.array(self.bin_counts),\n                   mea_bins=np.array(self.mea_bins), v_t=np.array(self.v_t), v_snap=np.array(self.v_snap),\n                   gate_t=np.array(self.gate_t), gates={k: np.array(v) for k, v in self.gates.items()},\n                   k_t=np.array(self.k_t), k_snap=np.array(self.k_snap), bal_t=np.array(self.bal_t),\n                   bal_exc=np.array(self.bal_exc), bal_inh=np.array(self.bal_inh), stim_label=list(self.stim_label),\n                   gate_ids=list(self.gate_ids), bin_ms=self.bin_ms)\n        return out\n\n\nclass BioNet:\n    def __init__(self, cfg, rng=None):\n        self.cfg = cfg\n        self.rng = rng or np.random.default_rng(cfg.seed)\n        self._build()\n        self.gel = GelMedium(cfg, self.xyz)\n        self.mea = HDMEA(cfg, self.xyz)\n        self._connect()\n        self.nrn = NeuronArray(self.types, dt=cfg.dt, fs_gM=cfg.fs_gM)\n        self.apply_environment()\n        N = self.N\n        self.g_ampa, self.g_nmda, self.g_gaba = np.zeros(N), np.zeros(N), np.zeros(N)\n        self.D = int(round(cfg.syn_delay / cfg.dt))\n        self.buf = np.zeros((max(self.D, 1), N), dtype=bool)\n        self.t = 0.0\n        self.step_i = 0\n        self.plasticity = None       # объект RSTDP (plasticity.py) или None\n        self.I_bias = np.zeros(N)\n\n    # ------------------------------------------------------------ сборка\n    def _build(self):\n        c, rng = self.cfg, self.rng\n        N = c.n_neurons\n        self.N = N\n        box = np.array(c.box)\n        self.box = box\n        self.xyz = rng.uniform([5, 5, 5], box - 5, size=(N, 3))\n        self.types = np.where(rng.random(N) < c.frac_fs, FS, RS)\n        self.is_exc = self.types == RS\n        self.is_inh = ~self.is_exc\n\n    def _connect(self):\n        \"\"\"p_ij = p0[класс] exp(-d/lambda) при d <= conn_radius (из вязкости геля) + n_long дальних\n        RS->RS аксонов с p ~ exp(-d/long_spread) (допущение, учебник гл. 4).\"\"\"\n        c, rng, N = self.cfg, self.rng, self.N\n        d = np.linalg.norm(self.xyz[:, None, :] - self.xyz[None, :, :], axis=2)\n        self.dist = d\n        R = self.gel.conn_radius\n        E, I = self.is_exc, self.is_inh\n        p0 = np.where(np.outer(E, E), c.p0_ee, 0.0) + np.where(np.outer(E, I), c.p0_ei, 0.0) + \\\n            np.where(np.outer(I, E), c.p0_ie, 0.0) + np.where(np.outer(I, I), c.p0_ii, 0.0)\n        p = p0 * np.exp(-d / c.conn_lambda) * (d <= R)\n        np.fill_diagonal(p, 0.0)\n        mask = rng.random((N, N)) < p\n        # дальние аксоны RS -> RS за пределами радиуса\n        exc_idx = np.where(E)[0]\n        self.long_mask = np.zeros((N, N), dtype=bool)\n        if c.n_long > 0:\n            for i in exc_idx:\n                cand = exc_idx[(d[i, exc_idx] > R)]\n                if len(cand) == 0:\n                    continue\n                w = np.exp(-d[i, cand] / c.long_spread)\n                k = min(c.n_long, len(cand))\n                tg = rng.choice(cand, size=k, replace=False, p=w / w.sum())\n                self.long_mask[i, tg] = True\n        mask |= self.long_mask\n        self.mask = mask\n        W = np.zeros((N, N))\n        wcls = np.where(np.outer(E, E), c.w_ee, 0.0) + np.where(np.outer(E, I), c.w_ei, 0.0) + \\\n            np.where(np.outer(I, E), c.w_ie, 0.0) + np.where(np.outer(I, I), c.w_ii, 0.0)\n        W[mask] = (wcls * rng.uniform(0.5, 1.5, (N, N)))[mask]\n        self.W = W\n        self.plastic = mask & np.outer(E, E)            # пластичны только E->E\n        self.W0 = W.copy()\n\n    def apply_environment(self):\n        \"\"\"Пересчитать температурные множители (после смены gel.T).\"\"\"\n        g, c = self.gel, self.cfg\n        self.nrn.set_temperature_factors(g.phi_gate, g.phi_cond)\n        dt = c.dt * g.phi_syn\n        self.dec_ampa, self.dec_nmda, self.dec_gaba = (np.exp(-dt / c.tau_ampa), np.exp(-dt / c.tau_nmda),\n                                                      np.exp(-dt / c.tau_gaba))\n        self.p_bg = 1.0 - np.exp(-c.bg_rate * c.dt / 1000.0)\n        self.bg_w = np.where(self.is_exc, c.bg_w, c.bg_w * c.bg_w_fs_mult)\n\n    def set_temperature(self, T):\n        self.gel.T = float(T)\n        self.apply_environment()\n\n    # ------------------------------------------------------------ симуляция\n    def run(self, dur_ms, I_ext=None, rec=None, learn=False):\n        \"\"\"Интегрирует сеть dur_ms. I_ext: ток (мкА/см2, вектор N) на весь кусок. Возвращает число спайков\n        каждого нейрона за кусок. learn: передавать спайки в правило пластичности (self.plasticity).\"\"\"\n        c, nrn, rng = self.cfg, self.nrn, self.rng\n        N, dt = self.N, c.dt\n        steps = int(round(dur_ms / dt))\n        I = (np.zeros(N) if I_ext is None else I_ext) + self.I_bias\n        counts = np.zeros(N)\n        k_every = max(1, int(round(c.k_update_ms / dt)))\n        k_acc = np.zeros(N)\n        bin_steps = int(round(rec.bin_ms / dt)) if rec else 0\n        v_steps = int(round(rec.v_every_ms / dt)) if (rec and rec.v_every_ms) else 0\n        kk_steps = int(round(rec.k_every_ms / dt)) if (rec and rec.k_every_ms) else 0\n        bin_acc = np.zeros(N)\n        bal_e = bal_i = 0.0\n        exc_w_rows = self.is_exc\n        plast = self.plasticity if learn else None\n        Wexc = self.W[exc_w_rows]\n        exc_rows = np.where(exc_w_rows)[0]\n        pos_in_exc = -np.ones(N, dtype=int)\n        pos_in_exc[exc_rows] = np.arange(len(exc_rows))\n        inh_rows = np.where(~exc_w_rows)[0]\n        pos_in_inh = -np.ones(N, dtype=int)\n        pos_in_inh[inh_rows] = np.arange(len(inh_rows))\n        Winh = self.W[inh_rows]\n        ek_shift = self.gel.ek_shift()\n        nrn.EK_shift = ek_shift if np.ndim(ek_shift) else np.zeros(N)\n        for s in range(steps):\n            # 1) спад синаптических проводимостей\n            self.g_ampa *= self.dec_ampa\n            self.g_nmda *= self.dec_nmda\n            self.g_gaba *= self.dec_gaba\n            # 2) пуассоновский фон (внешние AMPA-события)\n            bg = rng.random(N) < self.p_bg\n            if bg.any():\n                self.g_ampa += bg * self.bg_w\n            # 3) приход спайков, выпущенных syn_delay назад\n            slot = self.step_i % self.D\n            arr = self.buf[slot]\n            if arr.any():\n                idx = np.nonzero(arr)[0]\n                ie = pos_in_exc[idx]\n                ie = ie[ie >= 0]\n                if len(ie):\n                    inc = Wexc[ie].sum(axis=0)\n                    self.g_ampa += inc\n                    self.g_nmda += c.nmda_ratio * inc\n                ii = pos_in_inh[idx]\n                ii = ii[ii >= 0]\n                if len(ii):\n                    self.g_gaba += Winh[ii].sum(axis=0)\n            # 4) нейроны\n            B = self.gel.nmda_block(nrn.V)\n            gn = self.g_nmda * B\n            sp = nrn.step(I, self.g_ampa, gn, self.g_gaba, c.E_gaba)\n            self.buf[slot] = sp\n            self.t += dt\n            self.step_i += 1\n            if sp.any():\n                counts += sp\n                k_acc += sp\n                if plast is not None:\n                    plast.on_spikes(sp, self.t)\n            if plast is not None:\n                plast.decay_traces()\n            if (s + 1) % k_every == 0:\n                self.gel.k_step(k_acc, c.k_update_ms)\n                k_acc[:] = 0.0\n                ek_shift = self.gel.ek_shift()\n                if np.ndim(ek_shift):\n                    nrn.EK_shift = ek_shift\n            if rec is not None:\n                if rec.want_spikes and sp.any():\n                    ids = np.nonzero(sp)[0]\n                    rec.spk_t.extend([self.t] * len(ids))\n                    rec.spk_i.extend(ids.tolist())\n                bin_acc += sp\n                if rec.want_balance:\n                    # баланс E-I по ПРОВОДИМОСТЯМ на RS (как в in vivo оценках g_i/g_e, Destexhe et al. 2003);\n                    # по токам сравнивать нельзя: у ГАМК-А движущая сила V - E_GABA всего ~10 мВ у покоя\n                    bal_e += float((self.g_ampa + gn)[self.is_exc].mean())\n                    bal_i += float(self.g_gaba[self.is_exc].mean())\n                if bin_steps and (self.step_i % bin_steps == 0):\n                    rec.bins_t.append(self.t)\n                    rec.bin_counts.append([bin_acc[self.is_exc].sum(), bin_acc[self.is_inh].sum()])\n                    if rec.want_mea:\n                        rec.mea_bins.append(self.mea.A @ bin_acc)\n                    if rec.want_balance:\n                        rec.bal_t.append(self.t)\n                        rec.bal_exc.append(bal_e / bin_steps)\n                        rec.bal_inh.append(bal_i / bin_steps)\n                        bal_e = bal_i = 0.0\n                    bin_acc = np.zeros(N)\n                if v_steps and (self.step_i % v_steps == 0):\n                    rec.v_t.append(self.t)\n                    rec.v_snap.append(nrn.V.astype(np.float32).copy())\n                if rec.gate_ids and (self.step_i % rec.gate_every == 0):\n                    rec.gate_t.append(self.t)\n                    for k in (\"V\", \"m\", \"h\", \"n\", \"p\"):\n                        rec.gates[k].append(getattr(nrn, k)[rec.gate_ids].copy())\n                if kk_steps and (self.step_i % kk_steps == 0):\n                    rec.k_t.append(self.t)\n                    rec.k_snap.append(self.gel.K.mean(axis=2).copy())\n        return counts\n\n    # ------------------------------------------------------------ сводки\n    def summary(self):\n        E, I = self.is_exc, self.is_inh\n        k_in = self.mask.sum(axis=0)\n        return dict(N=self.N, n_RS=int(E.sum()), n_FS=int(I.sum()), synapses=int(self.mask.sum()),\n                    in_degree_mean=round(float(k_in.mean()), 1), long_axons=int(self.long_mask.sum()),\n                    conn_radius_um=round(self.gel.conn_radius, 1), electrodes=self.mea.n_el)\n", "plasticity.py": "\"\"\"Обучение: STDP -> след пригодности -> дофамин (R-STDP) + синаптическое масштабирование.\n\n1. STDP (парное правило с локальными следами; Bi & Poo 1998; Song, Miller & Abbott 2000):\n       x_i  — пресинаптический след:  dx/dt = -x/tau_plus,  x_i += 1 при спайке i\n       y_j  — постсинаптический след: dy/dt = -y/tau_minus, y_j += 1 при спайке j\n       спайк post j:  STDP_ij += A_plus  * x_i     (пре раньше пост -> потенциация)\n       спайк pre  i:  STDP_ij -= A_minus * y_j     (пост раньше пре -> депрессия)\n2. След пригодности (eligibility trace; Izhikevich 2007, Cereb. Cortex 17:2443):\n       de_ij/dt = -e_ij / tau_e + STDP_ij(t)\n   STDP не меняет вес сразу, а «помечает» синапс; метка гаснет за tau_e ~ 0.5–1 с.\n3. Дофамин (reward-modulated STDP; Frémaux & Gerstner 2016, обзор):\n       dW_ij = lr * DA * e_ij(t_награды),   DA = g_tonic * (R - R_ожид(стимул))\n   R = +1 за верный ответ, -1 за неверный; R_ожид — «критик», скользящее среднее награды для этого стимула\n   (Frémaux, Sprekeler & Gerstner 2010): без вычитания ожидания R-STDP дрейфует (та же ошибка, что в v2.1).\n   g_tonic — тонический дофамин (BioEnvironment.plasticity_gain из v3 или 1.0).\n4. Синаптическое масштабирование (гомеостаз; Turrigiano et al. 1998, Nature 391:892):\n       W_kj <- W_kj * (1 + eta * (r_target - r_j) / r_target)    для всех ВОЗБУЖДАЮЩИХ входов k нейрона j\n   r_j — частота нейрона, усреднённая по пробам. Тихий нейрон усиливает входы (защита от тишины),\n   разогнанный ослабляет (защита от эпилептического разгона). Мультипликативно: соотношения весов,\n   выученные R-STDP, сохраняются.\n\nРеализация следа без умножения матрицы N x N на каждом шаге: награда приходит в известный момент\nt_R, поэтому вклад события в момент t сразу пишется с весом exp(-(t_R - t)/tau_e) — это точное значение\ne(t_R). После выдачи награды весь след умножается на exp(-(t_R_next - t_R)/tau_e).\n\"\"\"\nimport numpy as np\n\n\nclass RSTDP:\n    def __init__(self, net, cfg):\n        self.net, self.cfg = net, cfg\n        N = net.N\n        self.x = np.zeros(N)\n        self.y = np.zeros(N)\n        self.e = np.zeros((N, N))\n        self.P = net.plastic                 # маска пластичных синапсов (E->E)\n        self.Pf = self.P.astype(float)\n        self.exc = net.is_exc\n        self.dx = np.exp(-cfg.dt / cfg.tau_plus)\n        self.dy = np.exp(-cfg.dt / cfg.tau_minus)\n        self.t_reward = cfg.t_pre + cfg.t_stim + cfg.decision_lag\n        self.critic = {}\n        self.rate_avg = np.where(net.is_exc, cfg.rate_target_rs, cfg.rate_target_fs).astype(float)\n        self.history = []\n\n    def decay_traces(self):\n        self.x *= self.dx\n        self.y *= self.dy\n\n    def on_spikes(self, sp, t):\n        c = self.cfg\n        w = np.exp(-(self.t_reward - t) / c.tau_elig) if t <= self.t_reward else 1.0\n        ids = np.nonzero(sp)[0]\n        # постсинаптические спайки: потенциация от недавних пре\n        self.e[:, ids] += (c.A_plus * w) * self.x[:, None] * self.Pf[:, ids]\n        # пресинаптические спайки (только RS): депрессия от недавних пост\n        pre = ids[self.exc[ids]]\n        if len(pre):\n            self.e[pre, :] -= (c.A_minus * w) * self.y[None, :] * self.Pf[pre, :]\n        self.x[ids] += 1.0\n        self.y[ids] += 1.0\n\n    def reward(self, R, stim_key, tonic_gain=1.0, next_reward_in=None):\n        \"\"\"Выдать награду R за пробу со стимулом stim_key. Возвращает фазический дофамин DA.\"\"\"\n        c, net = self.cfg, self.net\n        expected = self.critic.get(stim_key, 0.0)\n        DA = c.da_gain * tonic_gain * (R - expected)\n        self.critic[stim_key] = expected + c.critic_rate * (R - expected)\n        self.n_rewards = getattr(self, \"n_rewards\", 0) + 1\n        # затухание скорости обучения (опция): lr_eff = lr / (1 + n / lr_decay_trials); 0 = без затухания\n        lr = c.lr / (1.0 + self.n_rewards / c.lr_decay_trials) if c.lr_decay_trials else c.lr\n        if DA != 0.0:\n            net.W += lr * DA * self.e * self.Pf\n            np.clip(net.W, 0.0, None, out=net.W)\n            Wee = np.where(self.P, net.W, 0.0)\n            over = Wee > c.w_max_ee\n            net.W[over] = c.w_max_ee\n        if next_reward_in is not None:\n            self.e *= np.exp(-next_reward_in / c.tau_elig)\n            self.t_reward += next_reward_in\n        self.history.append(dict(R=R, expected=expected, DA=DA))\n        return DA\n\n    def scale(self, counts, dur_ms):\n        \"\"\"Синаптическое масштабирование после пробы: counts — спайки нейронов за пробу длиной dur_ms.\"\"\"\n        c, net = self.cfg, self.net\n        rate = counts / (dur_ms / 1000.0)\n        a = 1.0 / c.scale_tau_trials\n        self.rate_avg = (1 - a) * self.rate_avg + a * rate\n        if not c.scaling:\n            return np.ones(net.N)\n        target = np.where(net.is_exc, c.rate_target_rs, c.rate_target_fs)\n        f = 1.0 + c.scale_rate * (target - self.rate_avg) / target\n        f = np.clip(f, 1.0 - c.scale_clip, 1.0 + c.scale_clip)\n        net.W[self.exc, :] *= f[None, :]\n        np.minimum(net.W, np.where(self.P, c.w_max_ee, np.inf), out=net.W)\n        return f\n", "task_ab.py": "\"\"\"Задача дискриминации стимулов A/B на HD-MEA (по мотивам DishBrain: стимул -> ответ -> обратная связь).\n\nПроба (длительности в BioConfig):\n    t_pre   пауза без стимула\n    t_stim  стимул A или B: гауссово поле деполяризации от кластера электродов (mea.py), центр и амплитуда\n            с разбросом от пробы к пробе (stim_jitter_pos, stim_jitter_amp) — чтобы сеть не заучила один вход\n    lag     хвост окна решения\n    -> РЕШЕНИЕ: сигнал «моторных» кластеров электродов out_1 и out_2 (считывание MEA с затуханием) за окно\n       [начало стимула, конец стимула + lag]; ответ = кластер с большим сигналом\n    -> НАГРАДА: +1 за верный ответ, -1 за неверный -> фазический дофамин -> R-STDP (plasticity.py)\n    t_post  пауза после награды (след пригодности гаснет, сеть возвращается к фону)\n    -> синаптическое масштабирование по частотам за пробу\n\nПравильный ответ (mapping='straight'): A -> out_1, B -> out_2. Каждый выходной кластер равноудалён от A и B,\nпоэтому геометрия ответа не подсказывает: необученная сеть около 50 %, рост — заслуга обучения.\n(В пробном прогоне с асимметричной геометрией ответ был детерминирован расстоянием, и R-STDP с критиком\nне учился: при всегда одинаковом исходе R - R_ожид -> 0. Для обучения с подкреплением нужен разброс исходов.)\n\nОценка: отдельный ГСЧ (другой seed), без пластичности и без масштабирования, новые пробы — «отложенная» выборка.\n\"\"\"\nimport time\nimport numpy as np\nfrom .bionet import BioNet, Recorder\nfrom .plasticity import RSTDP\n\n\ndef dprime(x_a, x_b):\n    \"\"\"Индекс разделимости d' = (mu_A - mu_B) / sqrt((var_A + var_B) / 2).\"\"\"\n    x_a, x_b = np.asarray(x_a, float), np.asarray(x_b, float)\n    if len(x_a) < 2 or len(x_b) < 2:\n        return float(\"nan\")\n    s = np.sqrt((x_a.var(ddof=1) + x_b.var(ddof=1)) / 2.0)\n    return float((x_a.mean() - x_b.mean()) / s) if s > 0 else float(\"nan\")\n\n\nclass ABTask:\n    def __init__(self, cfg, net=None, env=None):\n        self.cfg = cfg\n        self.net = net or BioNet(cfg)\n        self.env = env                                  # BioEnvironment v3 (тонический дофамин) или None\n        self.rule = RSTDP(self.net, cfg)\n        self.net.plasticity = self.rule\n        m = self.net.mea\n        self.el_out = [m.electrodes_near(cfg.out_1, cfg.cluster_radius), m.electrodes_near(cfg.out_2, cfg.cluster_radius)]\n        self.el_in = [m.electrodes_near(cfg.stim_A, cfg.cluster_radius), m.electrodes_near(cfg.stim_B, cfg.cluster_radius)]\n        self.read_out = np.stack([m.A[e].sum(axis=0) for e in self.el_out])      # (2, N): вклад нейронов в кластер\n        self.read_in = np.stack([m.A[e].sum(axis=0) for e in self.el_in])\n        self.target = {\"A\": 1, \"B\": 0} if cfg.mapping == \"crossed\" else {\"A\": 0, \"B\": 1}\n        self.log = []\n        self.trial_i = 0\n        self.margin_var = 100.0                         # начальная оценка дисперсии запаса решения\n        self.sig_base = np.zeros(2)                     # скользящее среднее сигнала out1/out2 (center_decision)\n        self.period = cfg.t_pre + cfg.t_stim + cfg.decision_lag + cfg.t_post\n        self.net.run(200.0)          # прогрев: сеть уходит из покоя в фоновый режим\n\n    def stim_vector(self, label, rng):\n        c = self.cfg\n        cx, cy = c.stim_A if label == \"A\" else c.stim_B\n        cx, cy = cx + rng.normal(0, c.stim_jitter_pos), cy + rng.normal(0, c.stim_jitter_pos)\n        amp = c.stim_amp * (1.0 + rng.uniform(-c.stim_jitter_amp, c.stim_jitter_amp))\n        return self.net.mea.stim_current([(cx, cy)], amp)\n\n    def trial(self, label, learn=True, rng=None, rec=None):\n        \"\"\"Одна проба. Возвращает словарь: метка, ответ, верно ли, сигналы кластеров, дофамин.\"\"\"\n        c, net = self.cfg, self.net\n        rng = rng or net.rng\n        I = self.stim_vector(label, rng)\n        total = np.zeros(net.N)\n        t0 = net.t\n        if learn:   # момент награды этой пробы известен заранее: след пригодности пишется относительно него\n            t_r = net.t + c.t_pre + c.t_stim + c.decision_lag\n            if abs(self.rule.t_reward - t_r) > 1e-6:\n                self.rule.e *= np.exp(-max(t_r - self.rule.t_reward, 0.0) / c.tau_elig)\n                self.rule.t_reward = t_r\n        total += net.run(c.t_pre, None, rec, learn)\n        if rec is not None:\n            rec.stim_label.append((net.t, net.t + c.t_stim, label))\n        dec = net.run(c.t_stim, I, rec, learn)\n        dec += net.run(c.decision_lag, None, rec, learn)\n        total += dec\n        raw = self.read_out @ dec + rng.normal(0, c.mea_noise * np.sqrt(len(self.el_out[0])), 2)\n        sig_in = self.read_in @ dec\n        # центрирование решения: вычитаем скользящее среднее сигнала каждого кластера по пробам (обе метки).\n        # Аналог post - <post> в RM-Hebb v3: убирает общий дрейф «out1 сильнее out2», оставляя стимул-специфичную\n        # часть. Среднее обновляется только при обучении; в оценке порог заморожен.\n        sig = raw - self.sig_base if c.center_decision else raw\n        if learn and c.center_decision:\n            self.sig_base += c.center_rate * (raw - self.sig_base)\n        if sig[0] == sig[1]:\n            choice = int(rng.integers(2))\n        else:\n            choice = int(np.argmax(sig))\n        correct = choice == self.target[label]\n        # запас решения: сигнал «верного» кластера минус «неверного» (>0 — верно)\n        margin = float(sig[self.target[label]] - sig[1 - self.target[label]])\n        if c.reward_mode == \"margin\":\n            # градуированная награда (как shaping в Pong v3): R = tanh(запас / масштаб), масштаб = скользящее\n            # стандартное отклонение запаса. Меньше дисперсия, чем у ±1: сигнал есть и в «уверенно верных» пробах.\n            if learn:\n                self.margin_var = 0.98 * self.margin_var + 0.02 * margin ** 2\n            R = float(np.tanh(margin / max(np.sqrt(self.margin_var), 1e-9)))\n        else:\n            R = 1.0 if correct else -1.0\n        DA = 0.0\n        if learn:\n            tonic = self.env.plasticity_gain if self.env is not None else 1.0\n            DA = self.rule.reward(R, label, tonic, next_reward_in=self.period)\n            if self.env is not None:\n                self.env.step(c.t_stim, reward=DA)\n        total += net.run(c.t_post, None, rec, learn)\n        if learn:\n            self.rule.scale(total, net.t - t0)\n        rate_rs = total[net.is_exc].mean() / ((net.t - t0) / 1000.0)\n        rate_fs = total[net.is_inh].mean() / ((net.t - t0) / 1000.0)\n        out = dict(trial=self.trial_i, label=label, choice=choice, correct=bool(correct), R=R, DA=DA, margin=margin,\n                   out1=float(raw[0]), out2=float(raw[1]), dv=float(sig[self.target[\"A\"]] - sig[self.target[\"B\"]]),\n                   in_A=float(sig_in[0]), in_B=float(sig_in[1]), rate_RS=float(rate_rs), rate_FS=float(rate_fs),\n                   T=net.gel.T, K_mean=float(net.gel.K.mean()), w_ee_mean=float(net.W[net.plastic].mean()))\n        if learn:\n            self.trial_i += 1\n        return out\n\n    def train(self, n_trials, verbose_every=50, eval_every=None, eval_n=60, eval_seed=1000, callback=None):\n        \"\"\"Обучение n_trials проб (A/B в случайном порядке). Возвращает журнал проб и оценок.\"\"\"\n        rng = self.net.rng\n        evals = []\n        t0 = time.time()\n        for k in range(n_trials):\n            lab = \"A\" if rng.random() < 0.5 else \"B\"\n            r = self.trial(lab, learn=True)\n            self.log.append(r)\n            if verbose_every and (k + 1) % verbose_every == 0:\n                last = self.log[-verbose_every:]\n                print(f\"  проба {k+1:4d}: train-точность {100*np.mean([x['correct'] for x in last]):5.1f}% | \"\n                      f\"RS {np.mean([x['rate_RS'] for x in last]):.2f} Гц FS {np.mean([x['rate_FS'] for x in last]):.2f} Гц\"\n                      f\" | w_EE {r['w_ee_mean']:.5f} | {time.time()-t0:.0f} с\", flush=True)\n            if eval_every and (k + 1) % eval_every == 0:\n                ev = self.evaluate(eval_n, seed=eval_seed)\n                ev[\"trial\"] = k + 1\n                evals.append(ev)\n                print(f\"  оценка после {k+1}: {100*ev['acc']:.1f}% (d' {ev['dprime']:.2f})\", flush=True)\n            if callback:\n                callback(self, r)\n        return self.log, evals\n\n    def calibrate(self, n=40, seed=999):\n        \"\"\"Калибровка порога решения БЕЗ обучения: sig_base = средний сигнал out1/out2 за n проб.\n        Делается до оценки «до обучения», чтобы и она шла с центрированным решением (честное сравнение).\"\"\"\n        net = self.net\n        saved = self._save_dynamic()\n        rng = np.random.default_rng(seed)\n        old = net.rng\n        net.rng = rng\n        raws = []\n        try:\n            for k in range(n):\n                r = self.trial(\"A\" if k % 2 == 0 else \"B\", learn=False, rng=rng)\n                raws.append([r[\"out1\"], r[\"out2\"]])\n        finally:\n            net.rng = old\n            self._load_dynamic(saved)\n        self.sig_base = np.mean(raws, axis=0)\n        return self.sig_base\n\n    def evaluate(self, n=100, seed=1000, rec=None):\n        \"\"\"Отложенная оценка: n проб (поровну A/B) на отдельном ГСЧ, БЕЗ пластичности и масштабирования.\n        Состояние сети (V, проводимости, K+, время) восстанавливается после оценки.\"\"\"\n        net = self.net\n        saved = self._save_dynamic()\n        rng = np.random.default_rng(seed)\n        labels = np.array([\"A\", \"B\"] * (n // 2))\n        rng.shuffle(labels)\n        res = []\n        try:\n            net.rng, old_rng = rng, net.rng\n            for lab in labels:\n                res.append(self.trial(str(lab), learn=False, rng=rng, rec=rec))\n        finally:\n            net.rng = old_rng\n            self._load_dynamic(saved)\n        acc = float(np.mean([r[\"correct\"] for r in res]))\n        dv_a = [r[\"dv\"] for r in res if r[\"label\"] == \"A\"]\n        dv_b = [r[\"dv\"] for r in res if r[\"label\"] == \"B\"]\n        return dict(acc=acc, dprime=dprime(dv_a, dv_b), n=len(res), trials=res)\n\n    def _save_dynamic(self):\n        n, r = self.net, self.rule\n        return dict(V=n.nrn.V.copy(), m=n.nrn.m.copy(), h=n.nrn.h.copy(), nn=n.nrn.n.copy(), p=n.nrn.p.copy(),\n                    above=n.nrn.above.copy(), ga=n.g_ampa.copy(), gn=n.g_nmda.copy(), gg=n.g_gaba.copy(),\n                    buf=n.buf.copy(), t=n.t, si=n.step_i, K=n.gel.K.copy(), x=r.x.copy(), y=r.y.copy())\n\n    def _load_dynamic(self, d):\n        n, r = self.net, self.rule\n        n.nrn.V, n.nrn.m, n.nrn.h, n.nrn.n, n.nrn.p, n.nrn.above = d[\"V\"], d[\"m\"], d[\"h\"], d[\"nn\"], d[\"p\"], d[\"above\"]\n        n.g_ampa, n.g_nmda, n.g_gaba, n.buf, n.t, n.step_i = d[\"ga\"], d[\"gn\"], d[\"gg\"], d[\"buf\"], d[\"t\"], d[\"si\"]\n        n.gel.K, r.x, r.y = d[\"K\"], d[\"x\"], d[\"y\"]\n\n\ndef run_ab_experiment(seed=1, n_train=400, n_eval=100, eval_seed=None, verbose=True, **cfg_kw):\n    \"\"\"Полный опыт для одного seed: оценка до обучения -> обучение -> оценка после (отложенная выборка).\n    Возвращает словарь с точностями, d', временем расчёта и объектом задачи.\"\"\"\n    from .bioconfig import make_bio_config\n    cfg = make_bio_config(seed=seed, **cfg_kw)\n    t0 = time.time()\n    task = ABTask(cfg)\n    if cfg.center_decision:\n        task.calibrate()\n    es = 10_000 + seed if eval_seed is None else eval_seed\n    before = task.evaluate(n_eval, seed=es)\n    if verbose:\n        print(f\"[seed {seed}] до обучения: {100*before['acc']:.1f}% (d' {before['dprime']:.2f})\", flush=True)\n    W_before = task.net.W.copy()\n    task.train(n_train, verbose_every=50 if verbose else 0)\n    after = task.evaluate(n_eval, seed=es)\n    el = time.time() - t0\n    if verbose:\n        print(f\"[seed {seed}] после обучения: {100*after['acc']:.1f}% (d' {after['dprime']:.2f}), {el/60:.1f} мин\",\n              flush=True)\n    return dict(seed=seed, N=cfg.n_neurons, acc_before=before[\"acc\"], acc_after=after[\"acc\"],\n                dprime_before=before[\"dprime\"], dprime_after=after[\"dprime\"], n_train=n_train, n_eval=n_eval,\n                minutes=el / 60, task=task, W_before=W_before, eval_after=after, eval_before=before)\n", "info.py": "\"\"\"Информационные метрики: взаимная информация (MI) и энтропия переноса (TE), в битах.\n\nОценка «подстановочная» (plug-in) по гистограммам после дискретизации по квантилям. Она смещена вверх\nна малых выборках, поэтому рядом всегда печатается та же величина на ПЕРЕМЕШАННЫХ данных (суррогат):\nзначимо только превышение над суррогатом. Поправка = вычитание среднего по суррогатам.\n\nMI(X;Y) = sum p(x,y) log2 [ p(x,y) / (p(x) p(y)) ]\nTE(X->Y) = sum p(y_{t+1}, y_t, x_t) log2 [ p(y_{t+1} | y_t, x_t) / p(y_{t+1} | y_t) ]     (Schreiber 2000,\n           Phys. Rev. Lett. 85:461), история длины 1, сдвиг источника lag бинов.\nСуррогат для TE: источник берётся из ДРУГОЙ пробы (перестановка проб) — сохраняет автокорреляцию и\nстимул-зависимую форму сигнала, рвёт только связь внутри пробы.\n\"\"\"\nimport numpy as np\n\n\ndef quantize(x, n_levels=3):\n    \"\"\"Дискретизация по квантилям (равные доли) в 0..n_levels-1.\"\"\"\n    x = np.asarray(x, float)\n    if n_levels <= 1 or np.all(x == x.flat[0]):\n        return np.zeros(x.shape, dtype=int)\n    qs = np.quantile(x, np.linspace(0, 1, n_levels + 1)[1:-1])\n    return np.searchsorted(qs, x, side=\"right\").astype(int)\n\n\ndef _entropy_from_counts(c):\n    c = c[c > 0].astype(float)\n    p = c / c.sum()\n    return float(-(p * np.log2(p)).sum())\n\n\ndef _joint_entropy(*cols):\n    cols = [np.asarray(c, dtype=np.int64) for c in cols]\n    key = np.zeros(len(cols[0]), dtype=np.int64)\n    for c in cols:\n        key = key * (int(c.max()) + 1 if len(c) else 1) + c\n    _, cnt = np.unique(key, return_counts=True)\n    return _entropy_from_counts(cnt)\n\n\ndef mutual_information(x, y):\n    \"\"\"MI дискретных x, y (биты).\"\"\"\n    return _joint_entropy(x) + _joint_entropy(y) - _joint_entropy(x, y)\n\n\ndef mi_with_surrogate(x, y, n_shuffle=50, rng=None):\n    \"\"\"MI, среднее MI на перемешанном y и поправленная MI = MI - суррогат.\"\"\"\n    rng = rng or np.random.default_rng(0)\n    mi = mutual_information(x, y)\n    sur = [mutual_information(x, rng.permutation(y)) for _ in range(n_shuffle)]\n    return dict(MI=mi, MI_surrogate=float(np.mean(sur)), MI_corrected=mi - float(np.mean(sur)),\n                p_value=float((np.sum(np.array(sur) >= mi) + 1) / (n_shuffle + 1)))\n\n\ndef transfer_entropy(src_trials, dst_trials, lag=1):\n    \"\"\"TE(src -> dst) по набору проб (список 1D дискретных рядов одинаковой длины). Пары берутся только\n    внутри пробы (стык проб не используется).\"\"\"\n    yf, yp, xp = [], [], []\n    for x, y in zip(src_trials, dst_trials):\n        x, y = np.asarray(x), np.asarray(y)\n        T = len(y)\n        if T <= lag:\n            continue\n        yf.append(y[lag:])\n        yp.append(y[lag - 1:T - 1])\n        xp.append(x[:T - lag])\n    yf, yp, xp = np.concatenate(yf), np.concatenate(yp), np.concatenate(xp)\n    # TE = H(Yf,Yp) - H(Yp) - H(Yf,Yp,Xp) + H(Yp,Xp)\n    return (_joint_entropy(yf, yp) - _joint_entropy(yp) - _joint_entropy(yf, yp, xp) + _joint_entropy(yp, xp))\n\n\ndef te_with_surrogate(src_trials, dst_trials, lag=1, n_shuffle=30, rng=None):\n    rng = rng or np.random.default_rng(0)\n    te = transfer_entropy(src_trials, dst_trials, lag)\n    n = len(src_trials)\n    sur = []\n    for _ in range(n_shuffle):\n        perm = rng.permutation(n)\n        sur.append(transfer_entropy([src_trials[i] for i in perm], dst_trials, lag))\n    return dict(TE=te, TE_surrogate=float(np.mean(sur)), TE_corrected=te - float(np.mean(sur)),\n                p_value=float((np.sum(np.array(sur) >= te) + 1) / (n_shuffle + 1)))\n\n\ndef sensory_motor_info(mea_trials, el_sens, el_mot, labels, choices, n_levels=3, lag=1):\n    \"\"\"Сводка от сенсорных электродов к моторным.\n    mea_trials: список массивов (бины x электроды) по пробам; el_sens/el_mot: индексы электродов;\n    labels/choices: стимул и ответ в каждой пробе.\n    Возвращает: TE(сенсор->мотор), TE(мотор->сенсор) как контроль направления, MI(стимул; ответ),\n    MI(стимул; сигнал моторных электродов за пробу).\"\"\"\n    sens = [m[:, el_sens].sum(axis=1) for m in mea_trials]\n    mot = [m[:, el_mot].sum(axis=1) for m in mea_trials]\n    allS, allM = np.concatenate(sens), np.concatenate(mot)\n    qs = np.quantile(allS, np.linspace(0, 1, n_levels + 1)[1:-1])\n    qm = np.quantile(allM, np.linspace(0, 1, n_levels + 1)[1:-1])\n    sd = [np.searchsorted(qs, s, side=\"right\") for s in sens]\n    md = [np.searchsorted(qm, m, side=\"right\") for m in mot]\n    lab = np.array([0 if l == \"A\" else 1 for l in labels])\n    out = dict(TE_sens_to_mot=te_with_surrogate(sd, md, lag), TE_mot_to_sens=te_with_surrogate(md, sd, lag),\n               MI_stim_choice=mi_with_surrogate(lab, np.asarray(choices)))\n    return out\n", "viz.py": "\"\"\"Визуализация (Matplotlib): многопанельная панель V1–V6, сохранение PNG и анимации GIF/MP4.\n\nV1 3D-топология: нейроны X, Y, Z; форма — тип (RS кружок, FS треугольник); цвет — мембранный потенциал;\n   спайк — крупная белая обводка. Плоскость HD-MEA (Z = 0) — сетка точек, стимулирующие и моторные кластеры выделены.\nV2 растр E/I (RS синие, FS красные) + мгновенная частота сети (Гц, правая ось); серые полосы — стимулы.\nV3 тепловая карта HD-MEA (сигнал электродов в бине или за окно стимула).\nV4 веса W до/после обучения: средний вес между группами (у A, у B, у out_1, у out_2, остальные).\nV5 V(t) и затворы m, h, n, p выбранных нейронов (RS и FS).\nV6 баланс E–I (возбуждающий и тормозной токи на RS), кривая обучения и d', параметры среды, TE/MI.\n\"\"\"\nimport numpy as np\nimport matplotlib\nmatplotlib.use(\"Agg\")\nimport matplotlib.pyplot as plt\nfrom matplotlib import gridspec, animation\nfrom .bionet import Recorder\nfrom .neuron import RS, FS\n\nV_CMAP, V_RANGE = \"coolwarm\", (-80.0, 0.0)\nC_E, C_I = \"#2b6cb0\", \"#c53030\"\n\n\n# ------------------------------------------------------------------ сбор данных\ndef record_demo(task, labels=(\"A\", \"B\"), seed=4242, v_every_ms=2.0, gate_ids=None, bin_ms=5.0):\n    \"\"\"Записать несколько проб (без обучения) с полной телеметрией. Состояние сети восстанавливается.\"\"\"\n    net = task.net\n    if gate_ids is None:\n        r1 = task.read_out[0]\n        rs = np.where(net.is_exc)[0]\n        fs = np.where(net.is_inh)[0]\n        sel = np.abs(net.xyz[:, 0] - task.cfg.stim_A[0]) + np.abs(net.xyz[:, 1] - task.cfg.stim_A[1]) + net.xyz[:, 2]\n        gate_ids = [int(rs[np.argmin(sel[rs])]), int(fs[np.argmin(sel[fs])])]\n    rec = Recorder(spikes=True, bin_ms=bin_ms, v_every_ms=v_every_ms, gate_ids=gate_ids, k_every_ms=10.0)\n    saved = task._save_dynamic()\n    rng = np.random.default_rng(seed)\n    old = net.rng\n    net.rng = rng\n    t_start = net.t\n    trials = []\n    try:\n        for lab in labels:\n            trials.append(task.trial(lab, learn=False, rng=rng, rec=rec))\n    finally:\n        net.rng = old\n        task._load_dynamic(saved)\n    d = rec.arrays()\n    d[\"t0\"] = t_start\n    d[\"trials\"] = trials\n    return d\n\n\ndef record_mea_trials(task, n=100, seed=31337):\n    \"\"\"Пробы без обучения с записью только MEA-бинов: материал для TE/MI. Возвращает (mea_trials, labels, choices).\"\"\"\n    net = task.net\n    saved = task._save_dynamic()\n    rng = np.random.default_rng(seed)\n    old = net.rng\n    net.rng = rng\n    labels = np.array([\"A\", \"B\"] * (n // 2))\n    rng.shuffle(labels)\n    out, ch = [], []\n    try:\n        for lab in labels:\n            rec = Recorder(spikes=False, bin_ms=5.0, mea=True, balance=False)\n            r = task.trial(str(lab), learn=False, rng=rng, rec=rec)\n            out.append(np.array(rec.mea_bins))\n            ch.append(r[\"choice\"])\n    finally:\n        net.rng = old\n        task._load_dynamic(saved)\n    return out, list(labels), ch\n\n\nGROUP_NAMES = [\"у A\", \"у B\", \"у out1\", \"у out2\", \"прочие\"]\n\n\ndef neuron_groups(task, radius=80.0):\n    \"\"\"Группа каждого RS-нейрона по близости (в плоскости XY) к центрам кластеров; FS не входят (-1).\"\"\"\n    c, net = task.cfg, task.net\n    g = np.full(net.N, 4)\n    for k, ctr in enumerate([c.stim_A, c.stim_B, c.out_1, c.out_2]):\n        d = np.hypot(net.xyz[:, 0] - ctr[0], net.xyz[:, 1] - ctr[1])\n        g[(d <= radius) & (g == 4)] = k\n    g[~net.is_exc] = -1\n    return g\n\n\ndef group_weight_matrix(task, W, groups=None):\n    groups = neuron_groups(task) if groups is None else groups\n    M = task.net.mask\n    out = np.full((5, 5), np.nan)\n    for a in range(5):\n        for b in range(5):\n            sel = M[np.ix_(groups == a, groups == b)]\n            if sel.any():\n                out[a, b] = W[np.ix_(groups == a, groups == b)][sel].mean()\n    return out\n\n\n# ------------------------------------------------------------------ панели\ndef _v1(ax, task, V, spiking=None, title=\"V1 3D-топология\"):\n    net, c = task.net, task.cfg\n    xyz = net.xyz\n    m = task.net.mea\n    ax.scatter(m.exy[:, 0], m.exy[:, 1], np.zeros(m.n_el), s=2, c=\"#999999\", alpha=0.25, depthshade=False)\n    for el, col in [(task.el_in[0], \"#38a169\"), (task.el_in[1], \"#d69e2e\"), (task.el_out[0], \"#805ad5\"),\n                    (task.el_out[1], \"#dd6b20\")]:\n        ax.scatter(m.exy[el, 0], m.exy[el, 1], np.zeros(len(el)), s=14, c=col, depthshade=False, marker=\"s\")\n    arts = []\n    for t, mk in [(RS, \"o\"), (FS, \"^\")]:\n        sel = net.types == t\n        sizes = np.full(sel.sum(), 7.0)\n        edge = np.full(sel.sum(), 0.0)\n        if spiking is not None:\n            sizes = np.where(spiking[sel], 40.0, 7.0)\n            edge = np.where(spiking[sel], 1.2, 0.0)\n        sc = ax.scatter(xyz[sel, 0], xyz[sel, 1], xyz[sel, 2], c=V[sel], cmap=V_CMAP, vmin=V_RANGE[0], vmax=V_RANGE[1],\n                        s=sizes, marker=mk, edgecolors=\"white\", linewidths=edge, depthshade=False)\n        arts.append((sc, sel))\n    ax.set_xlim(0, c.box[0]); ax.set_ylim(0, c.box[1]); ax.set_zlim(0, c.box[2])\n    ax.set_xlabel(\"X, мкм\", fontsize=7); ax.set_ylabel(\"Y, мкм\", fontsize=7); ax.set_zlabel(\"Z, мкм\", fontsize=7)\n    ax.tick_params(labelsize=6)\n    ax.set_title(title, fontsize=9)\n    ax.view_init(elev=22, azim=-60)\n    ax.text2D(0.0, -0.12, \"электроды Z=0: ■A зел. ■B жёлт. ■out1 фиол. ■out2 оранж.; ● RS ▲ FS; белая обводка = спайк\",\n              transform=ax.transAxes, fontsize=6)\n    return arts\n\n\ndef _v2(ax, rec, net, t0=0.0):\n    t, i = rec[\"spk_t\"] - t0, rec[\"spk_i\"]\n    if len(t):\n        e = net.is_exc[i]\n        ax.scatter(t[e], i[e], s=0.6, c=C_E, marker=\"|\", linewidths=0.4)\n        ax.scatter(t[~e], i[~e], s=0.6, c=C_I, marker=\"|\", linewidths=0.4)\n    for a, b, lab in rec[\"stim_label\"]:\n        ax.axvspan(a - t0, b - t0, color=\"0.85\", zorder=0)\n        ax.text((a + b) / 2 - t0, net.N * 1.01, lab, ha=\"center\", fontsize=7)\n    ax.set_xlabel(\"t, мс\", fontsize=7); ax.set_ylabel(\"нейрон\", fontsize=7)\n    ax.set_title(\"V2 растр E (синие) / I (красные) и частота сети\", fontsize=9)\n    ax.tick_params(labelsize=6)\n    ax2 = ax.twinx()\n    bt = rec[\"bins_t\"] - t0\n    bc = rec[\"bin_counts\"]\n    if len(bc):\n        rate_e = bc[:, 0] / net.is_exc.sum() / (rec[\"bin_ms\"] / 1000.0)\n        rate_i = bc[:, 1] / net.is_inh.sum() / (rec[\"bin_ms\"] / 1000.0)\n        ax2.plot(bt, rate_e, color=C_E, lw=1.0, label=\"RS, Гц\")\n        ax2.plot(bt, rate_i, color=C_I, lw=1.0, label=\"FS, Гц\")\n        ax2.legend(fontsize=6, loc=\"upper right\")\n    ax2.set_ylabel(\"частота, Гц\", fontsize=7)\n    ax2.tick_params(labelsize=6)\n    ax.set_xlim(0, max(bt.max() if len(bt) else 1.0, 1.0))\n    return ax2\n\n\ndef _v3(ax, task, img, title=\"V3 HD-MEA\"):\n    m = task.net.mea\n    p = task.cfg.mea_pitch\n    im = ax.imshow(img.T, origin=\"lower\", cmap=\"magma\", extent=[0, m.nx * p, 0, m.ny * p], aspect=\"equal\")\n    for ctr, lab in [(task.cfg.stim_A, \"A\"), (task.cfg.stim_B, \"B\"), (task.cfg.out_1, \"out1\"), (task.cfg.out_2, \"out2\")]:\n        ax.add_patch(plt.Circle(ctr, task.cfg.cluster_radius, fill=False, color=\"cyan\", lw=0.8))\n        ax.text(ctr[0], ctr[1] + task.cfg.cluster_radius + 8, lab, color=\"cyan\", fontsize=7, ha=\"center\")\n    ax.set_title(title, fontsize=9)\n    ax.set_xlabel(\"X, мкм\", fontsize=7); ax.set_ylabel(\"Y, мкм\", fontsize=7)\n    ax.tick_params(labelsize=6)\n    return im\n\n\ndef _v4(fig, sub, task, W_before, W_after):\n    g = neuron_groups(task)\n    mb, ma = group_weight_matrix(task, W_before, g), group_weight_matrix(task, W_after, g)\n    vmax = np.nanmax([mb, ma])\n    inner = gridspec.GridSpecFromSubplotSpec(1, 2, subplot_spec=sub, wspace=0.35)\n    ims = []\n    for k, (M, ttl) in enumerate([(mb, \"до\"), (ma, \"после\")]):\n        ax = fig.add_subplot(inner[k])\n        ims.append(ax.imshow(M * 1000, cmap=\"viridis\", vmin=0, vmax=vmax * 1000))\n        ax.set_xticks(range(5)); ax.set_yticks(range(5))\n        ax.set_xticklabels(GROUP_NAMES, rotation=60, fontsize=6); ax.set_yticklabels(GROUP_NAMES, fontsize=6)\n        ax.set_xlabel(\"пост\", fontsize=7)\n        if k == 0:\n            ax.set_ylabel(\"пре\", fontsize=7)\n        ax.set_title(f\"V4 W(E→E) {ttl}, мкСм/см²\", fontsize=8)\n        for a in range(5):\n            for b in range(5):\n                if not np.isnan(M[a, b]):\n                    ax.text(b, a, f\"{M[a,b]*1000:.1f}\", ha=\"center\", va=\"center\", fontsize=5.5, color=\"white\")\n    return ims\n\n\ndef _v5(fig, sub, rec, t0=0.0, net=None):\n    \"\"\"V(t) и затворы: для каждого выбранного нейрона две оси (V сверху, затворы снизу). У FS g_M = 0,\n    поэтому p не рисуется (переменная считается, но тока не даёт).\"\"\"\n    n = min(2, len(rec[\"gate_ids\"]))\n    inner = gridspec.GridSpecFromSubplotSpec(2 * n, 1, subplot_spec=sub, hspace=0.12,\n                                             height_ratios=[1.3, 1.0] * n)\n    gt = rec[\"gate_t\"] - t0\n    axes = []\n    for k in range(n):\n        gid = rec[\"gate_ids\"][k]\n        is_fs = net is not None and net.types[gid] == FS\n        nm = \"FS\" if is_fs else \"RS\"\n        axv = fig.add_subplot(inner[2 * k])\n        axv.plot(gt, rec[\"gates\"][\"V\"][:, k], color=\"k\", lw=0.7)\n        axv.set_ylabel(f\"{nm} #{gid}\\nV, мВ\", fontsize=7)\n        axv.tick_params(labelsize=6); axv.set_xticklabels([])\n        axg = fig.add_subplot(inner[2 * k + 1], sharex=axv)\n        for gname, col in [(\"m\", \"#e53e3e\"), (\"h\", \"#3182ce\"), (\"n\", \"#38a169\"), (\"p\", \"#d69e2e\")]:\n            if gname == \"p\" and is_fs:\n                continue\n            axg.plot(gt, rec[\"gates\"][gname][:, k], color=col, lw=0.7, label=gname)\n        axg.set_ylim(-0.02, 1.02); axg.tick_params(labelsize=6)\n        axg.legend(fontsize=5.5, ncol=4, loc=\"upper right\", framealpha=0.6)\n        axg.set_ylabel(\"затворы\", fontsize=7)\n        if k == 0:\n            axv.set_title(\"V5 V(t) и затворы m, h, n, p (у FS g_M = 0)\", fontsize=9)\n            axg.set_xticklabels([])\n        else:\n            axg.set_xlabel(\"t, мс\", fontsize=7)\n        axes += [axv, axg]\n    return axes\n\n\ndef _v6(fig, sub, task, rec, train_log, evals, info, t0=0.0):\n    inner = gridspec.GridSpecFromSubplotSpec(1, 3, subplot_spec=sub, wspace=0.45)\n    # баланс E-I\n    ax = fig.add_subplot(inner[0])\n    if len(rec[\"bal_t\"]):\n        ax.plot(rec[\"bal_t\"] - t0, np.array(rec[\"bal_exc\"]) * 1000, color=C_E, lw=0.8, label=\"g_E (AMPA+NMDA·B)\")\n        ax.plot(rec[\"bal_t\"] - t0, np.array(rec[\"bal_inh\"]) * 1000, color=C_I, lw=0.8, label=\"g_I (ГАМК-А)\")\n        ratio = np.mean(rec[\"bal_inh\"]) / max(np.mean(rec[\"bal_exc\"]), 1e-12)\n        ax.set_title(f\"V6 баланс E–I на RS: g_I/g_E = {ratio:.2f}\", fontsize=8)\n        for a, b, lab in rec[\"stim_label\"]:\n            ax.axvspan(a - t0, b - t0, color=\"0.9\", zorder=0)\n    ax.set_xlabel(\"t, мс\", fontsize=7); ax.set_ylabel(\"мкСм/см² (среднее по RS)\", fontsize=7)\n    ax.legend(fontsize=6); ax.tick_params(labelsize=6)\n    # обучение и d'\n    ax = fig.add_subplot(inner[1])\n    if train_log:\n        corr = np.array([r[\"correct\"] for r in train_log], float)\n        w = max(1, min(50, len(corr)))\n        roll = np.convolve(corr, np.ones(w) / w, mode=\"valid\") * 100\n        ax.plot(np.arange(w, len(corr) + 1), roll, color=\"0.5\", lw=0.8, label=f\"обучение (окно {w})\")\n    if evals:\n        ax.plot([e[\"trial\"] for e in evals], [100 * e[\"acc\"] for e in evals], \"o-\", color=\"#2f855a\", ms=3,\n                label=\"отложенная оценка\")\n        axd = ax.twinx()\n        axd.plot([e[\"trial\"] for e in evals], [e[\"dprime\"] for e in evals], \"s--\", color=\"#805ad5\", ms=3)\n        axd.set_ylabel(\"d'\", fontsize=7, color=\"#805ad5\"); axd.tick_params(labelsize=6)\n    ax.axhline(50, color=\"r\", ls=\":\", lw=0.7); ax.axhline(70, color=\"g\", ls=\":\", lw=0.7)\n    ax.set_ylim(20, 100)\n    ax.set_xlabel(\"проба\", fontsize=7); ax.set_ylabel(\"точность, %\", fontsize=7)\n    ax.set_title(\"точность A/B и d'\", fontsize=8); ax.legend(fontsize=5.5, loc=\"lower right\"); ax.tick_params(labelsize=6)\n    # среда + TE/MI\n    ax = fig.add_subplot(inner[2])\n    labels, vals, surs = [], [], []\n    if info:\n        for key, nm, f in [(\"TE_sens_to_mot\", \"TE с→м\", \"TE\"), (\"TE_mot_to_sens\", \"TE м→с\", \"TE\"),\n                           (\"MI_stim_choice\", \"MI стим;ответ\", \"MI\")]:\n            if key in info:\n                labels.append(nm); vals.append(info[key][f]); surs.append(info[key][f + \"_surrogate\"])\n    if labels:\n        x = np.arange(len(labels))\n        ax.bar(x - 0.18, vals, 0.36, color=\"#3182ce\", label=\"данные\")\n        ax.bar(x + 0.18, surs, 0.36, color=\"0.7\", label=\"суррогат\")\n        ax.set_xticks(x); ax.set_xticklabels(labels, fontsize=6, rotation=20)\n        ax.legend(fontsize=6)\n    ax.set_ylabel(\"бит\", fontsize=7); ax.tick_params(labelsize=6)\n    g = task.net.gel\n    c = task.cfg\n    ax.set_title(\"TE/MI и среда\", fontsize=8)\n    txt = (f\"T = {g.T:.1f} °C (φ затворов ×{g.phi_gate:.2f})\\n[Mg²⁺] = {g.mg:.1f} мМ\\n\"\n           f\"вязкость ×{c.viscosity:.2f} → R связн. {g.conn_radius:.0f} мкм\\n\"\n           f\"[K⁺]o ср. {g.K.mean():.2f} / макс {g.K.max():.2f} мМ\\nфон {c.bg_rate:.0f} Гц × {c.bg_w*1000:.1f} мкСм/см²\")\n    ax.text(1.04, 0.98, txt, transform=ax.transAxes, fontsize=6, va=\"top\", family=\"monospace\")\n\n\ndef dashboard(task, W_before, rec, train_log=None, evals=None, info=None, path=None, frame_idx=None):\n    \"\"\"Собрать панель V1–V6 и (если path) сохранить PNG. Возвращает фигуру.\"\"\"\n    net = task.net\n    t0 = rec.get(\"t0\", 0.0)\n    fig = plt.figure(figsize=(19, 12.5))\n    gs = gridspec.GridSpec(3, 3, figure=fig, height_ratios=[1.15, 1.0, 0.9], hspace=0.42, wspace=0.28)\n    # V1: снимок в момент наибольшей активности стимула\n    if frame_idx is None and len(rec[\"v_t\"]):\n        frame_idx = _peak_frame(rec, net)\n    V = rec[\"v_snap\"][frame_idx] if len(rec[\"v_t\"]) else net.nrn.V\n    sp = _spiking_near(rec, rec[\"v_t\"][frame_idx], net.N) if len(rec[\"v_t\"]) else None\n    ax1 = fig.add_subplot(gs[0, 0], projection=\"3d\")\n    arts = _v1(ax1, task, V, sp, title=f\"V1 3D-топология, t = {rec['v_t'][frame_idx]-t0:.0f} мс\" if len(rec['v_t']) else \"V1\")\n    cb = fig.colorbar(arts[0][0], ax=ax1, shrink=0.55, pad=0.08)\n    cb.set_label(\"V, мВ\", fontsize=7); cb.ax.tick_params(labelsize=6)\n    ax2 = fig.add_subplot(gs[0, 1:])\n    _v2(ax2, rec, net, t0)\n    # V3: средний сигнал за окно стимула A (первая проба)\n    ax3 = fig.add_subplot(gs[1, 0])\n    img = _mea_window(rec, task, 0)\n    im = _v3(ax3, task, img, title=\"V3 HD-MEA: сигнал за окно стимула \" + (rec[\"stim_label\"][0][2] if rec[\"stim_label\"] else \"\"))\n    fig.colorbar(im, ax=ax3, shrink=0.8).ax.tick_params(labelsize=6)\n    _v4(fig, gs[1, 1], task, W_before, net.W)\n    _v5(fig, gs[1, 2], rec, t0, net)\n    _v6(fig, gs[2, :], task, rec, train_log or [], evals or [], info or {}, t0)\n    s = net.summary()\n    fig.suptitle(f\"OI-биочип v4: N = {s['N']} (RS {s['n_RS']}, FS {s['n_FS']}), синапсов {s['synapses']}, \"\n                 f\"электродов {s['electrodes']}, задача A/B\", fontsize=11)\n    if path:\n        fig.savefig(path, dpi=110, bbox_inches=\"tight\")\n    return fig\n\n\ndef _peak_frame(rec, net):\n    if not len(rec[\"stim_label\"]):\n        return len(rec[\"v_t\"]) // 2\n    a, b, _ = rec[\"stim_label\"][0]\n    vt = rec[\"v_t\"]\n    st = rec[\"spk_t\"]\n    best, best_n = 0, -1\n    for k, t in enumerate(vt):\n        if a <= t <= b:\n            n = np.sum((st > t - 2.0) & (st <= t))\n            if n > best_n:\n                best, best_n = k, n\n    return best\n\n\ndef _spiking_near(rec, t, N, win=2.0):\n    st, si = rec[\"spk_t\"], rec[\"spk_i\"]\n    sp = np.zeros(N, dtype=bool)\n    sel = (st > t - win) & (st <= t)\n    sp[si[sel]] = True\n    return sp\n\n\ndef _mea_window(rec, task, k):\n    if not len(rec[\"mea_bins\"]):\n        return np.zeros((task.net.mea.nx, task.net.mea.ny))\n    a, b, _ = rec[\"stim_label\"][k] if rec[\"stim_label\"] else (rec[\"bins_t\"][0], rec[\"bins_t\"][-1], \"\")\n    sel = (rec[\"bins_t\"] > a) & (rec[\"bins_t\"] <= b + task.cfg.decision_lag)\n    return task.net.mea.to_image(rec[\"mea_bins\"][sel].sum(axis=0))\n\n\n# ------------------------------------------------------------------ анимация\ndef animate(task, rec, path_gif=None, path_mp4=None, max_frames=120, fps=12):\n    \"\"\"Анимация: V1 (потенциалы и спайки в 3D), V2 (бегущий курсор), V3 (MEA в текущем бине), V5 (курсор).\n    Сохраняет GIF (Pillow) и/или MP4 (ffmpeg, если есть). Возвращает список сохранённых путей.\"\"\"\n    net = task.net\n    t0 = rec.get(\"t0\", 0.0)\n    vt = rec[\"v_t\"]\n    idx = np.linspace(0, len(vt) - 1, min(max_frames, len(vt))).astype(int)\n    fig = plt.figure(figsize=(18, 9))\n    gs = gridspec.GridSpec(2, 3, figure=fig, width_ratios=[1.15, 1.0, 0.8], hspace=0.38, wspace=0.3)\n    ax1 = fig.add_subplot(gs[:, 0], projection=\"3d\")\n    arts = _v1(ax1, task, rec[\"v_snap\"][idx[0]], _spiking_near(rec, vt[idx[0]], net.N))\n    cb = fig.colorbar(arts[0][0], ax=ax1, shrink=0.45, pad=0.06)\n    cb.set_label(\"V, мВ\", fontsize=7); cb.ax.tick_params(labelsize=6)\n    ax2 = fig.add_subplot(gs[0, 1])\n    _v2(ax2, rec, net, t0)\n    cursors = [ax2.axvline(0, color=\"k\", lw=1.0)]\n    ax3 = fig.add_subplot(gs[0, 2])\n    mb, bt = rec[\"mea_bins\"], rec[\"bins_t\"]\n    vmax = np.percentile(mb, 99.5) if len(mb) else 1.0\n    im = _v3(ax3, task, task.net.mea.to_image(mb[0]) if len(mb) else np.zeros((2, 2)), title=\"V3 HD-MEA, бин 5 мс\")\n    im.set_clim(0, max(vmax, 1e-6))\n    for a in _v5(fig, gs[1, 1], rec, t0, net):\n        cursors.append(a.axvline(0, color=\"k\", lw=0.8))\n    ax6 = fig.add_subplot(gs[1, 2])\n    if len(rec[\"bal_t\"]):\n        ax6.plot(rec[\"bal_t\"] - t0, np.array(rec[\"bal_exc\"]) * 1000, color=C_E, lw=0.8, label=\"g_E\")\n        ax6.plot(rec[\"bal_t\"] - t0, np.array(rec[\"bal_inh\"]) * 1000, color=C_I, lw=0.8, label=\"g_I\")\n        ax6.legend(fontsize=6)\n    ax6.set_title(\"V6 баланс E–I на RS, мкСм/см²\", fontsize=9); ax6.tick_params(labelsize=6)\n    ax6.set_xlabel(\"t, мс\", fontsize=7)\n    cursors.append(ax6.axvline(0, color=\"k\", lw=0.8))\n    title = fig.suptitle(\"\", fontsize=11)\n\n    def update(k):\n        f = idx[k]\n        t = vt[f]\n        sp = _spiking_near(rec, t, net.N)\n        for sc, sel in arts:\n            sc.set_array(rec[\"v_snap\"][f][sel])\n            sc.set_sizes(np.where(sp[sel], 40.0, 7.0))\n            sc.set_linewidths(np.where(sp[sel], 1.2, 0.0))\n        for cu in cursors:\n            cu.set_xdata([t - t0, t - t0])\n        if len(bt):\n            j = int(np.clip(np.searchsorted(bt, t), 0, len(bt) - 1))\n            im.set_data(task.net.mea.to_image(mb[j]).T)\n        lab = \"\"\n        for a, b, l in rec[\"stim_label\"]:\n            if a <= t <= b:\n                lab = f\" | стимул {l}\"\n        title.set_text(f\"t = {t - t0:6.1f} мс{lab}\")\n        return []\n\n    anim = animation.FuncAnimation(fig, update, frames=len(idx), interval=1000 / fps, blit=False)\n    saved = []\n    if path_gif:\n        anim.save(path_gif, writer=animation.PillowWriter(fps=fps), dpi=70)\n        saved.append(path_gif)\n    if path_mp4 and animation.writers.is_available(\"ffmpeg\"):\n        anim.save(path_mp4, writer=animation.FFMpegWriter(fps=fps, bitrate=1800), dpi=90)\n        saved.append(path_mp4)\n    plt.close(fig)\n    return saved\n\n\ndef fi_figure(fi_rs, fi_fs, traces=None, path=None):\n    \"\"\"f–I кривые RS и FS (средняя, первая и последняя мгновенная частота) + примеры трасс.\"\"\"\n    fig, axes = plt.subplots(1, 3 if traces else 1, figsize=(15 if traces else 5, 4))\n    axes = np.atleast_1d(axes)\n    ax = axes[0]\n    for rows, col, nm in [(fi_rs, C_E, \"RS\"), (fi_fs, C_I, \"FS\")]:\n        I = [r[\"I_uA_cm2\"] for r in rows]\n        ax.plot(I, [r[\"rate_Hz\"] for r in rows], \"o-\", color=col, label=f\"{nm} средняя\")\n        ax.plot(I, [r[\"f_first_Hz\"] for r in rows], \"--\", color=col, alpha=0.6, label=f\"{nm} 1/ISI первый\")\n        ax.plot(I, [r[\"f_last_Hz\"] for r in rows], \":\", color=col, alpha=0.9, label=f\"{nm} 1/ISI последний\")\n    ax.set_xlabel(\"ток, мкА/см²\"); ax.set_ylabel(\"частота, Гц\"); ax.legend(fontsize=7); ax.set_title(\"f–I (1 с ступенька)\")\n    if traces:\n        for k, (nm, tr) in enumerate(traces.items()):\n            inner = axes[k + 1].get_subplotspec().subgridspec(2, 1, hspace=0.08, height_ratios=[1.3, 1.0])\n            axes[k + 1].remove()\n            av = fig.add_subplot(inner[0])\n            ag = fig.add_subplot(inner[1], sharex=av)\n            av.plot(tr[\"t\"], tr[\"V\"], color=\"k\", lw=0.7)\n            av.set_title(nm, fontsize=9); av.set_ylabel(\"V, мВ\"); av.tick_params(labelbottom=False)\n            for g, col in [(\"m\", \"#e53e3e\"), (\"h\", \"#3182ce\"), (\"n\", \"#38a169\"), (\"p\", \"#d69e2e\")]:\n                if g == \"p\" and nm.startswith(\"FS\"):\n                    continue\n                ag.plot(tr[\"t\"], tr[g], color=col, lw=0.7, label=g)\n            ag.set_ylim(-0.02, 1.02); ag.set_xlabel(\"t, мс\"); ag.set_ylabel(\"затворы\")\n            ag.legend(fontsize=7, ncol=4, loc=\"upper right\")\n            av.set_xlim(0, 350)\n    if path:\n        fig.savefig(path, dpi=110, bbox_inches=\"tight\")\n    return fig\n"}
for name, text in FILES.items():
    open(f'oipong/{name}', 'w', encoding='utf-8').write(text)
open('oi_v4_checks.py', 'w', encoding='utf-8').write("\"\"\"Проверочные прогоны v4 (то, что записано в README, раздел «Что проверено запуском»).\n\n    python run_checks.py fi                         # (а) f–I одиночных RS/FS, адаптация, шаг 0.05 vs 0.025 мс\n    python run_checks.py ab --seeds 1 2 3 --train 900 --eval 200   # (б) A/B при N=1000, сиды параллельно\n    python run_checks.py viz --seed 1               # (в) панель V1–V6 (PNG) + анимация (GIF/MP4) + TE/MI\n    python run_checks.py env --seed 1               # среда: температура, Mg2+, вязкость (на обученной сети)\n\nРезультаты: out/*.json, out/*.png, out/*.gif, out/*.mp4. Обученные веса: out/ab_seed<N>.npz.\n\"\"\"\nimport argparse\nimport json\nimport os\nimport sys\nimport time\n\nimport numpy as np\n\nHERE = os.path.dirname(os.path.abspath(__file__))\nsys.path.insert(0, HERE)\nOUT = os.environ.get(\"OI_OUT\", os.path.join(HERE, \"out\"))   # каталог результатов (переменная OI_OUT)\nos.makedirs(OUT, exist_ok=True)\n\n\ndef _dump(name, obj):\n    with open(os.path.join(OUT, name), \"w\") as f:\n        json.dump(obj, f, ensure_ascii=False, indent=1, default=float)\n\n\n# ----------------------------------------------------------------------------- (а)\ndef check_fi():\n    from oipong.neuron import RS, FS, fi_curve, simulate_single\n    t0 = time.time()\n    cur = [0.25, 0.5, 0.75, 1.0, 1.5, 2.0, 3.0, 4.0, 6.0, 8.0]\n    res = {}\n    for dt in (0.025, 0.05, 0.1):\n        res[dt] = {nm: fi_curve(t, cur, dt=dt) for nm, t in ((\"RS\", RS), (\"FS\", FS))}\n    print(\"ток мкА/см2 (нА RS) | RS Гц (1/ISI первый -> последний) | FS Гц (первый -> последний) | RS/FS Гц при dt 0.025 и 0.1\")\n    for k, I in enumerate(cur):\n        r, f = res[0.05][\"RS\"][k], res[0.05][\"FS\"][k]\n        print(f\"  {I:5.2f} ({r['I_nA']:.3f}) | RS {r['rate_Hz']:6.1f} ({r['f_first_Hz']:6.1f} -> {r['f_last_Hz']:6.1f}) \"\n              f\"| FS {f['rate_Hz']:6.1f} ({f['f_first_Hz']:6.1f} -> {f['f_last_Hz']:6.1f}) \"\n              f\"| {res[0.025]['RS'][k]['rate_Hz']:.0f}/{res[0.025]['FS'][k]['rate_Hz']:.0f}, \"\n              f\"{res[0.1]['RS'][k]['rate_Hz']:.0f}/{res[0.1]['FS'][k]['rate_Hz']:.0f}\")\n    dev = max(abs(res[d][nm][k][\"rate_Hz\"] - res[0.025][nm][k][\"rate_Hz\"]) / max(res[0.025][nm][k][\"rate_Hz\"], 1.0)\n              for d in (0.05, 0.1) for nm in (\"RS\", \"FS\") for k in range(len(cur)))\n    print(f\"макс. относительное расхождение частоты с dt = 0.025 мс: {100*dev:.1f}%\")\n    traces = {}\n    for nm, t in ((\"RS, 1.5 мкА/см² (адаптация)\", RS), (\"FS, 1.5 мкА/см² (без адаптации)\", FS)):\n        _, tr = simulate_single(t, 1.5, t_ms=600.0, delay_ms=50.0, record=True)\n        traces[nm] = tr\n    try:\n        from oipong import viz\n        viz.fi_figure(res[0.05][\"RS\"], res[0.05][\"FS\"], traces, path=os.path.join(OUT, \"fi_curves.png\"))\n        print(\"сохранено out/fi_curves.png\")\n    except ImportError as e:\n        print(\"matplotlib нет, рисунок не строился:\", e)\n    _dump(\"fi.json\", {str(k): v for k, v in res.items()})\n    print(f\"время: {time.time()-t0:.1f} с\")\n\n\n# ----------------------------------------------------------------------------- (б)\ndef _ab_worker(args):\n    seed, n_train, n_eval, eval_every, kw = args\n    os.environ.setdefault(\"OMP_NUM_THREADS\", \"1\")\n    from oipong.bioconfig import make_bio_config, bio_cfg_to_dict\n    from oipong.task_ab import ABTask\n    t0 = time.time()\n    cfg = make_bio_config(seed=seed, **kw)\n    task = ABTask(cfg)\n    if cfg.center_decision:\n        task.calibrate()                    # порог решения до обучения (без пластичности)\n    es = 10_000 + seed                      # ГСЧ отложенной оценки: не пересекается с обучением\n    before = task.evaluate(n_eval, seed=es)\n    W_before = task.net.W.copy()\n    evals = [dict(trial=0, acc=before[\"acc\"], dprime=before[\"dprime\"])]\n    done = 0\n    while done < n_train:\n        k = min(eval_every, n_train - done)\n        task.train(k, verbose_every=0)\n        done += k\n        ev = task.evaluate(n_eval, seed=es)\n        evals.append(dict(trial=done, acc=ev[\"acc\"], dprime=ev[\"dprime\"]))\n        print(f\"[seed {seed}] проба {done}: отложенная оценка {100*ev['acc']:.1f}% d' {ev['dprime']:.2f} \"\n              f\"| {(time.time()-t0)/60:.1f} мин\", flush=True)\n    # независимая повторная оценка на другом ГСЧ\n    ev2 = task.evaluate(n_eval, seed=es + 500)\n    np.savez_compressed(os.path.join(OUT, f\"ab_seed{seed}.npz\"), W=task.net.W, W_before=W_before,\n                        sig_base=task.sig_base)\n    log = [{k: v for k, v in r.items()} for r in task.log]\n    _dump(f\"ab_seed{seed}_log.json\", dict(cfg=bio_cfg_to_dict(cfg), evals=evals, train_log=log))\n    return dict(seed=seed, N=cfg.n_neurons, n_train=n_train, n_eval=n_eval, acc_before=before[\"acc\"],\n                acc_after=evals[-1][\"acc\"], acc_after_seed2=ev2[\"acc\"], dprime_before=before[\"dprime\"],\n                dprime_after=evals[-1][\"dprime\"], minutes=(time.time() - t0) / 60, evals=evals)\n\n\ndef check_ab(seeds, n_train, n_eval, eval_every, procs, kw):\n    from multiprocessing import Pool\n    t0 = time.time()\n    jobs = [(s, n_train, n_eval, eval_every, kw) for s in seeds]\n    if procs > 1:\n        with Pool(min(procs, len(jobs))) as p:\n            res = p.map(_ab_worker, jobs)\n    else:\n        res = [_ab_worker(j) for j in jobs]\n    print(\"\\nseed | N | до обучения | после (оценка 1) | после (оценка 2, другой ГСЧ) | d' до -> после | мин\")\n    for r in res:\n        print(f\"  {r['seed']} | {r['N']} | {100*r['acc_before']:.1f}% | {100*r['acc_after']:.1f}% | \"\n              f\"{100*r['acc_after_seed2']:.1f}% | {r['dprime_before']:.2f} -> {r['dprime_after']:.2f} | {r['minutes']:.1f}\")\n    acc = np.array([r[\"acc_after\"] for r in res])\n    print(f\"среднее после: {100*acc.mean():.1f}% (min {100*acc.min():.1f}%), всего {(time.time()-t0)/60:.1f} мин\")\n    _dump(\"ab_results.json\", dict(results=res, kw=kw, wall_minutes=(time.time() - t0) / 60))\n\n\ndef _load_trained(seed):\n    from oipong.bioconfig import make_bio_config\n    from oipong.task_ab import ABTask\n    d = json.load(open(os.path.join(OUT, f\"ab_seed{seed}_log.json\")))\n    cfg_d = d[\"cfg\"]\n    cfg = make_bio_config(**{k: (tuple(v) if isinstance(v, list) else v) for k, v in cfg_d.items()})\n    task = ABTask(cfg)                      # геометрия и маска связей детерминированы seed\n    z = np.load(os.path.join(OUT, f\"ab_seed{seed}.npz\"))\n    assert task.net.W.shape == z[\"W\"].shape\n    task.net.W[:] = z[\"W\"]\n    if \"sig_base\" in z:\n        task.sig_base = z[\"sig_base\"].copy()\n    return task, z[\"W_before\"], d\n\n\n# ----------------------------------------------------------------------------- (в)\ndef check_viz(seed):\n    from oipong import viz\n    from oipong.info import sensory_motor_info\n    t0 = time.time()\n    task, W_before, d = _load_trained(seed)\n    mea_trials, labels, choices = viz.record_mea_trials(task, n=120)\n    els = np.concatenate(task.el_in)\n    elm = np.concatenate(task.el_out)\n    info = sensory_motor_info(mea_trials, els, elm, labels, choices)\n    print(\"TE сенсор->мотор: %.4f бит (суррогат %.4f, p=%.3f); TE мотор->сенсор: %.4f (суррогат %.4f, p=%.3f)\" % (\n        info[\"TE_sens_to_mot\"][\"TE\"], info[\"TE_sens_to_mot\"][\"TE_surrogate\"], info[\"TE_sens_to_mot\"][\"p_value\"],\n        info[\"TE_mot_to_sens\"][\"TE\"], info[\"TE_mot_to_sens\"][\"TE_surrogate\"], info[\"TE_mot_to_sens\"][\"p_value\"]))\n    print(\"MI(стимул; ответ): %.4f бит (суррогат %.4f, p=%.3f), точность на этих пробах %.1f%%\" % (\n        info[\"MI_stim_choice\"][\"MI\"], info[\"MI_stim_choice\"][\"MI_surrogate\"], info[\"MI_stim_choice\"][\"p_value\"],\n        100 * np.mean([(c == task.target[l]) for c, l in zip(choices, labels)])))\n    rec = viz.record_demo(task, labels=(\"A\", \"B\"))\n    evals = d[\"evals\"]\n    viz.dashboard(task, W_before, rec, d[\"train_log\"], evals, info, path=os.path.join(OUT, \"dashboard.png\"))\n    print(\"сохранено out/dashboard.png\")\n    saved = viz.animate(task, rec, path_gif=os.path.join(OUT, \"dashboard_anim.gif\"),\n                        path_mp4=os.path.join(OUT, \"dashboard_anim.mp4\"), max_frames=110)\n    print(\"сохранено:\", \", \".join(os.path.relpath(p, HERE) for p in saved))\n    _dump(\"info_metrics.json\", info)\n    print(f\"время: {time.time()-t0:.1f} с\")\n\n\ndef check_env(seed, n=60):\n    \"\"\"Обученная сеть при другой температуре, [Mg2+] и вязкости (вязкость меняет сборку, поэтому для неё\n    печатается только радиус связности и число синапсов новой сети).\"\"\"\n    from oipong.bioconfig import make_bio_config\n    from oipong.bionet import BioNet\n    task, _, _ = _load_trained(seed)\n    rows = []\n    for T in (30.0, 33.0, 36.0, 39.0):\n        task.net.set_temperature(T)\n        ev = task.evaluate(n, seed=777)\n        r = np.mean([x[\"rate_RS\"] for x in ev[\"trials\"]])\n        rows.append(dict(kind=\"T\", value=T, acc=ev[\"acc\"], rate_RS=float(r)))\n        print(f\"T = {T:4.1f} °C: точность {100*ev['acc']:.1f}%, RS {r:.2f} Гц\")\n    task.net.set_temperature(36.0)\n    for mg in (0.0, 1.0, 2.0):\n        task.net.gel.mg = mg\n        ev = task.evaluate(n, seed=777)\n        r = np.mean([x[\"rate_RS\"] for x in ev[\"trials\"]])\n        rows.append(dict(kind=\"Mg\", value=mg, acc=ev[\"acc\"], rate_RS=float(r)))\n        print(f\"[Mg2+] = {mg:.1f} мМ: точность {100*ev['acc']:.1f}%, RS {r:.2f} Гц\")\n    task.net.gel.mg = 1.0\n    for eta in (0.5, 1.0, 2.0, 4.0):\n        net = BioNet(make_bio_config(seed=seed, viscosity=eta, n_neurons=task.cfg.n_neurons))\n        s = net.summary()\n        rows.append(dict(kind=\"viscosity\", value=eta, conn_radius=s[\"conn_radius_um\"], synapses=s[\"synapses\"]))\n        print(f\"вязкость x{eta}: радиус связности {s['conn_radius_um']} мкм, синапсов {s['synapses']}\")\n    _dump(\"env_checks.json\", rows)\n\n\nif __name__ == \"__main__\":\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\"what\", choices=[\"fi\", \"ab\", \"viz\", \"env\"])\n    ap.add_argument(\"--seeds\", type=int, nargs=\"+\", default=[1, 2, 3])\n    ap.add_argument(\"--seed\", type=int, default=1)\n    ap.add_argument(\"--train\", type=int, default=900)\n    ap.add_argument(\"--eval\", type=int, default=200)\n    ap.add_argument(\"--eval-every\", type=int, default=300)\n    ap.add_argument(\"--procs\", type=int, default=4)\n    ap.add_argument(\"--n\", type=int, default=1000)\n    ap.add_argument(\"--kw\", type=str, default=\"{}\", help=\"переопределения BioConfig, JSON\")\n    a = ap.parse_args()\n    if a.what == \"fi\":\n        check_fi()\n    elif a.what == \"ab\":\n        kw = json.loads(a.kw)\n        kw[\"n_neurons\"] = a.n\n        check_ab(a.seeds, a.train, a.eval, a.eval_every, a.procs, kw)\n    elif a.what == \"viz\":\n        check_viz(a.seed)\n    else:\n        check_env(a.seed)\n")
open('oi_engine.py', 'w', encoding='utf-8').write('"""Совместимость со старыми ячейками: from oi_engine import *"""\nimport json, pickle, time\nimport numpy as np\nimport pandas as pd\nfrom oipong import *\nfrom oipong.config import *\nfrom oipong import BOX\n')
for m in [m for m in list(sys.modules) if m in ('oipong', 'oi_engine', 'oi_v4_checks') or m.startswith('oipong.')]:
    del sys.modules[m]
if '' not in sys.path: sys.path.insert(0, '')
import oipong
print('Записано:', ', '.join(FILES), '+ oi_v4_checks.py + oi_engine.py; версия', oipong.__version__)


In [ ]:
# Ячейка 2: параметры опыта уровня 2 (нейроны Pospischil, HD-MEA, гель, R-STDP). Меняйте здесь.
from oipong import make_bio_config, bio_cfg_to_dict
N_NEURONS = 1000          # >= 1000 по ТЗ; время ~ линейно по N (замер: README)
SEEDS = [1, 2, 3]         # в Colab 2 ядра: сиды считаются по 2 параллельно
N_TRAIN = 900             # проб обучения на сид
N_EVAL = 200              # проб отложенной оценки (без обучения, отдельный ГСЧ)
EVAL_EVERY = 300
CFG_OVERRIDES = dict(     # переопределения BioConfig, например:
    # temperature=36.0, mg_mM=1.0, viscosity=1.0, lr=..., stim_amp=...
)
cfg = make_bio_config(n_neurons=N_NEURONS, **CFG_OVERRIDES)
print({k: v for k, v in bio_cfg_to_dict(cfg).items() if k in ('n_neurons', 'box', 'frac_fs', 'temperature', 'mg_mM', 'viscosity', 'lr', 'tau_elig', 'stim_amp', 'mapping')})


In [ ]:
# Ячейка 3: (а) одиночные нейроны RS и FS: f–I кривая, адаптация RS, проверка шага dt (~1-2 мин)
import oi_v4_checks as chk
chk.check_fi()
from IPython.display import Image, display
display(Image('out/fi_curves.png'))


In [ ]:
# Ячейка 4: (б) обучение A/B при N >= 1000 и отложенная оценка по сидам. Долго: см. замер времени в README.
import json, os
import oi_v4_checks as chk
kw = dict(CFG_OVERRIDES, n_neurons=N_NEURONS)
chk.check_ab(SEEDS, N_TRAIN, N_EVAL, EVAL_EVERY, procs=os.cpu_count() or 1, kw=kw)


In [ ]:
# Ячейка 5: (в) панель V1–V6 (PNG) + анимация (GIF/MP4) + TE/MI на обученной сети сида VIZ_SEED
import oi_v4_checks as chk
from IPython.display import Image, Video, display
VIZ_SEED = SEEDS[0]
chk.check_viz(VIZ_SEED)
display(Image('out/dashboard.png'))
if os.path.exists('out/dashboard_anim.mp4'):
    display(Video('out/dashboard_anim.mp4', embed=True))
else:
    display(Image('out/dashboard_anim.gif'))


In [ ]:
# Ячейка 6: среда на обученной сети: температура (Q10), [Mg2+] (блок NMDA), вязкость -> радиус связности
import oi_v4_checks as chk
chk.check_env(SEEDS[0])


## Pong v3 (движок v3 без изменений)
Ниже ячейки 2–8 ноутбука v3 как есть: конфигурация, диагностика, обучение Pong, 3D-визуализация plotly, графики, выгрузка в Google Таблицу, химия и температура. Пакет из ячейки 1 содержит все модули v3, поэтому `from oi_engine import *` работает.

In [ ]:
# Ячейка 2: конфигурация эксперимента. Меняйте только здесь; ячейки 3-4 читают experiment.json.
import json
from oi_engine import PRESETS

EXPERIMENT = dict(
    name="v3_direct",                # имя в журнале запусков
    preset="direct",                 # "direct" | "hidden" | "hidden_structural" | "two_layer" (НЕ учится) | "legacy_v21"
    cfg=dict(
        seed=1,
        # lr=0.15,                   # скорость обучения (при rule='rmhebb' устойчива в диапазоне 0.15-0.5)
        # n_hid0=14, hid_max=36,     # размер скрытого слоя (для hidden*)
        # temperature=37.0,          # °C (Q10-масштабирование кинетики)
        # da_tonic=1.0,              # тонический дофамин (0 = пластичность выключена)
        # glu_tonic=1.0, gaba_tonic=1.0,
        # env_enabled=True,          # False = полностью выключить химико-термическую среду
    ),
)
assert EXPERIMENT["preset"] in PRESETS, f"preset должен быть одним из {list(PRESETS)}"
with open("experiment.json", "w") as f:
    json.dump(EXPERIMENT, f, indent=2, ensure_ascii=False)
print("Сохранено experiment.json:", EXPERIMENT)

In [ ]:
# Ячейка 3: диагностика (запускать до длинного обучения). Ничего не сохраняет в чекпойнт.
import json
from oi_engine import *

exp = json.load(open("experiment.json"))
cfg = make_config(exp["preset"], **exp["cfg"])

print("--- Базовые уровни без сети (2000 эпизодов) ---")
for pol in ["random", "stay", "perfect"]:
    print(f"  {pol:8s}: {100*baseline_winrate(cfg, pol):5.1f}%")

print("\n--- Необученная сеть (200 эпизодов, без обучения) ---")
print(f"  win = {100*OIPong(cfg).evaluate(200)['win']:.1f}%")

print("\n--- Оракул: веса выставлены вручную (потолок считывания) ---")
print(f"  win = {100*make_oracle(cfg).evaluate(200)['win']:.1f}%  (ожидается ~100 %; если ниже ~95 %, чините кодирование/моторный слой)")

RUN_LADDER = False         # True: ступенчатая отладка (оракул -> direct -> скрытый слой -> нейрогенез -> температура), ~7 мин
if RUN_LADDER:
    print("\n--- Ступенчатая отладка ---")
    print(debug_ladder(episodes=300, eval_n=80).to_string(index=False))

RUN_TEMPERATURE_SCAN = False   # True: как температура меняет частоту спайков и win rate оракула (~1 мин)
if RUN_TEMPERATURE_SCAN:
    print(temperature_scan().to_string(index=False))

RUN_DOPAMINE_SCAN = False      # True: скорость обучения при разном тоническом дофамине (~2 мин)
if RUN_DOPAMINE_SCAN:
    print(dopamine_scan().to_string(index=False))

RUN_BENCHMARK = False      # True: обучение с нуля на 3 seed (разброс между запусками)
if RUN_BENCHMARK:
    df, summ = benchmark(seeds=(1, 2, 3), episodes=600, eval_every=200, eval_n=100,
                         preset=exp["preset"], **{k: v for k, v in exp["cfg"].items() if k != "seed"})
    print("\nСводка по seed (mean / std / min / max, %):")
    print(summ.to_string())

RUN_ABLATION = False       # True: сравнить варианты на 3 seed (долго)
if RUN_ABLATION:
    table = ablation({
        "direct (v3)":                 dict(preset="direct"),
        "direct, старое правило v2.1": dict(preset="legacy_v21"),
        "со скрытым слоем":            dict(preset="hidden"),
        "скрытый + нейрогенез":        dict(preset="hidden_structural"),
        "без среды":                   dict(preset="direct", env_enabled=False),
    }, seeds=(1, 2, 3), episodes=600, eval_every=200)
    print(table.to_string())

In [ ]:
# Ячейка 4: обучение до цели по ОЦЕНКЕ (без обучения и без шума исследования). Повторный запуск дообучает.
import os, json, warnings
import pandas as pd
warnings.filterwarnings("ignore")
from oi_engine import *

RESUME = True            # True: продолжить с oi_state.pkl (конфиг берётся из чекпойнта), False: с нуля
MAX_EPISODES = 3000      # эпизодов В ЭТОМ ЗАПУСКЕ; None = без ограничения (может не сойтись никогда)
TARGET = 1.0             # доля побед в оценке; 1.0 = 100 %
EVAL_EVERY, EVAL_N, CONFIRM_N = 100, 100, 300   # цель: EVAL_N из EVAL_N, затем подтверждение на CONFIRM_N новых

exp = json.load(open("experiment.json"))
if RESUME and os.path.exists("oi_state.pkl"):
    game = load_game("oi_state.pkl")
    print(f"Продолжаем: эпизод {game.episode}, нейронов {game.N} (конфиг из чекпойнта; experiment.json игнорируется)")
else:
    for f in ["oi_episodes.csv", "oi_frames.csv", "oi_events.csv", "oi_eval.csv"]:
        if os.path.exists(f): os.remove(f)
    game = OIPong(make_config(exp["preset"], **exp["cfg"]))
    print("Новая сеть:", exp["name"], "нейронов", game.N)

base_random = baseline_winrate(game.cfg, "random")
ep_df, fr_df, ev_df, reached = game.train(target=TARGET, max_episodes=MAX_EPISODES, eval_every=EVAL_EVERY,
                                          eval_n=EVAL_N, confirm_n=CONFIRM_N, log_every=EVAL_EVERY * 2,
                                          frame_log_every=20)

def append_csv(df, path):
    if len(df): df.to_csv(path, mode="a", header=not os.path.exists(path), index=False)

append_csv(ep_df, "oi_episodes.csv")
append_csv(fr_df, "oi_frames.csv")          # покадровые данные: только CSV, в Google Таблицу не отправляются
append_csv(ev_df, "oi_eval.csv")
EVENT_COLS = ["Episode", "Event", "Neuron_ID", "Note", "Hidden", "Total"]
pd.DataFrame(game.events, columns=EVENT_COLS).to_csv("oi_events.csv", index=False)   # с заголовком, даже если событий нет
game.neurons_table().to_csv("oi_neurons.csv", index=False)
game.synapses_table().to_csv("oi_synapses.csv", index=False)
save_game(game, "oi_state.pkl")
save_config(game.cfg, "oi_cfg_used.json")
json.dump(dict(baseline_random_pct=round(100 * base_random, 2)), open("oi_run_meta.json", "w"))
log_run(dict(name=exp["name"], preset=exp["preset"], episodes_total=game.episode, reached=bool(reached),
             last_eval_pct=float(ev_df["Eval_Win_Pct"].iloc[-1]), baseline_random_pct=round(100 * base_random, 2),
             neurons=int(game.N), synapses=int(game.mask.sum()), cfg=cfg_to_dict(game.cfg)))
print("Сохранено: oi_state.pkl, oi_episodes.csv, oi_eval.csv, oi_frames.csv, oi_events.csv, oi_neurons.csv, oi_synapses.csv, oi_runs.jsonl")

# Структура выученных прямых весов: «верные» пути (мяч выше -> UP) должны быть сильнее «неверных»
import numpy as np
S, up = game.idx(ROLE_SENS), game.mot_up
if game.mask[np.ix_(S, up)].any():
    ok, bad = weight_structure(game)
    print(f"Прямые пути сенсор->мотор: верные {ok:.3f} | неверные {bad:.3f} (чем больше разрыв, тем лучше выучено)")

In [ ]:
# Ячейка 5: 3D-биосеть и игра (БЕЗ обучения, без шума исследования). Все нейроны отрисованы всегда.
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from oi_engine import *

N_EPISODES = 3
game = load_game("oi_state.pkl")

rec = []
for _ in range(N_EPISODES):
    hit, err, fr, _ = game.play_episode(learn=False, record=True)
    for f in fr:
        f["hit"] = hit
    rec.extend(fr)

role_color = {ROLE_SENS: "#00E5FF", ROLE_HID: "#3388FF", ROLE_MOT: "#FF2255"}
base_colors = [role_color[int(r)] for r in game.role]
xyz = game.xyz
labels = [f"N{int(game.nid[i])} {ROLE_NAME[int(game.role[i])]}" for i in range(game.N)]

ii, jj = np.where(game.mask)
w = game.W[ii, jj]
groups = []
if len(w):
    q1, q2 = np.quantile(w, [0.33, 0.66])
    for lo, hi, width, alpha in [(-1, q1, 1, 0.20), (q1, q2, 2, 0.35), (q2, 99, 4, 0.60)]:
        sel = (w > lo) & (w <= hi)
        ex, ey, ez = [], [], []
        for a, b in zip(ii[sel], jj[sel]):
            ex += [xyz[a, 0], xyz[b, 0], None]; ey += [xyz[a, 1], xyz[b, 1], None]; ez += [xyz[a, 2], xyz[b, 2], None]
        groups.append(go.Scatter3d(x=ex, y=ey, z=ez, mode="lines", hoverinfo="skip",
                                   line=dict(color=f"rgba(0,240,180,{alpha})", width=width),
                                   name=f"Синапсы (w≤{hi:.2f})" if hi < 99 else "Сильные синапсы"))

def ghost_trace():
    # статичный слой: нейроны видны всегда, даже если анимация не отрисовала кадр
    return go.Scatter3d(x=xyz[:, 0], y=xyz[:, 1], z=xyz[:, 2], mode="markers", text=labels, hoverinfo="text",
                        marker=dict(size=5, color=base_colors, opacity=1.0), name="Нейроны (всегда)")

def node_trace(f):
    sp = f["spikes"] > 0
    colors = ["#FFFFFF" if sp[i] else base_colors[i] for i in range(game.N)]
    sizes = [14 if sp[i] else 7 for i in range(game.N)]
    text = [f"{labels[i]}<br>V={f['v'][i]:.1f} мВ<br>спайков в кадре: {int(f['spikes'][i])}" for i in range(game.N)]
    return go.Scatter3d(x=xyz[:, 0], y=xyz[:, 1], z=xyz[:, 2], mode="markers", text=text, hoverinfo="text",
                        marker=dict(size=sizes, color=colors, opacity=1.0, line=dict(width=0)), name="Активность")

def ball_trace(f):
    return go.Scatter(x=[f["ball_x"]], y=[f["ball_y"]], mode="markers", marker=dict(size=16, color="yellow"), name="Мяч")

def paddle_trace(f):
    return go.Scatter(x=[1.0, 1.0], y=[f["paddle_y"] - game.cfg.hit_half, f["paddle_y"] + game.cfg.hit_half],
                      mode="lines", line=dict(color="magenta", width=10), name="Ракетка")

fig = make_subplots(rows=1, cols=2, specs=[[{"type": "scene"}, {"type": "xy"}]],
                    subplot_titles=("3D-биочип (голубые - вход, синие - скрытые, красные - выход, белые - спайк)", "Pong"),
                    column_widths=[0.62, 0.38])
for g_ in groups:
    fig.add_trace(g_, row=1, col=1)
n_edge = len(groups)
fig.add_trace(ghost_trace(), row=1, col=1)
fig.add_trace(node_trace(rec[0]), row=1, col=1)
fig.add_trace(ball_trace(rec[0]), row=1, col=2)
fig.add_trace(paddle_trace(rec[0]), row=1, col=2)
i_nodes, i_ball, i_pad = n_edge + 1, n_edge + 2, n_edge + 3

fig.frames = [go.Frame(data=[node_trace(f), ball_trace(f), paddle_trace(f)],
                       traces=[i_nodes, i_ball, i_pad], name=str(k)) for k, f in enumerate(rec)]

fig.update_layout(
    template="plotly_dark", height=620, margin=dict(l=10, r=10, t=60, b=10),
    scene=dict(xaxis=dict(range=[0, BOX[0]], title="X мкм"), yaxis=dict(range=[0, BOX[1]], title="Y мкм"),
               zaxis=dict(range=[0, BOX[2]], title="Z мкм"), aspectmode="data"),
    xaxis2=dict(range=[-0.05, 1.1], title="X"), yaxis2=dict(range=[-0.05, 1.05], title="Y"),
    updatemenus=[dict(type="buttons", showactive=False, x=0.02, y=-0.02, buttons=[
        dict(label="▶ Играть", method="animate",
             args=[None, dict(frame=dict(duration=150, redraw=True), fromcurrent=True, transition=dict(duration=0))]),
        dict(label="⏸ Пауза", method="animate",
             args=[[None], dict(frame=dict(duration=0, redraw=False), mode="immediate")])])],
    sliders=[dict(steps=[dict(method="animate", label=str(k),
                              args=[[str(k)], dict(mode="immediate", frame=dict(duration=0, redraw=True))])
                         for k in range(len(rec))], x=0.15, len=0.8)],
    title=f"Эпизод обучения {game.episode} | нейронов: {game.N} | синапсов: {int(game.mask.sum())}",
)
fig.show()

In [ ]:
# Ячейка 6: панель графиков процесса обучения (читает CSV, ничего не пересчитывает)
import os, json
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def read_csv_safe(path):
    """Пустой или отсутствующий файл -> пустая таблица (а не EmptyDataError)."""
    if not os.path.exists(path) or os.path.getsize(path) == 0:
        return pd.DataFrame()
    try:
        return pd.read_csv(path)
    except pd.errors.EmptyDataError:
        return pd.DataFrame()

ep = read_csv_safe("oi_episodes.csv")
assert len(ep), "oi_episodes.csv пуст или не найден: сначала запустите ячейку 4"
ev = read_csv_safe("oi_events.csv")
ed = read_csv_safe("oi_eval.csv")
meta = json.load(open("oi_run_meta.json")) if os.path.exists("oi_run_meta.json") else {}

fig = make_subplots(rows=3, cols=2, subplot_titles=(
    "Оценка без обучения (главная метрика) и win rate во время обучения, %", "Популяция нейронов",
    "Средний вес синапсов по типам", "Активность (доля кадров со спайком)",
    "Расстояние промаха (скользящее среднее 20)", "Число синапсов"))

if len(ed):
    fig.add_trace(go.Scatter(x=ed.Episode, y=ed.Eval_Win_Pct, name="оценка (без обучения)", mode="lines+markers",
                             line=dict(color="limegreen", width=3)), 1, 1)
fig.add_trace(go.Scatter(x=ep.Episode, y=ep.WinRate_Rolling_Pct, name="во время обучения (окно 30, с шумом исследования)",
                         line=dict(color="gray", dash="dot")), 1, 1)
if "baseline_random_pct" in meta:
    fig.add_hline(y=meta["baseline_random_pct"], line_dash="dash", line_color="red", row=1, col=1,
                  annotation_text="случайная ракетка")

fig.add_trace(go.Scatter(x=ep.Episode, y=ep.Neurons_Total, name="всего", line=dict(color="cyan")), 1, 2)
fig.add_trace(go.Scatter(x=ep.Episode, y=ep.Neurons_Hidden, name="скрытых", line=dict(color="dodgerblue")), 1, 2)
if len(ev) and "Event" in ev:
    for kind, color, sym in [("neurogenesis", "lime", "triangle-up"), ("apoptosis", "red", "triangle-down")]:
        e = ev[ev.Event == kind]
        if len(e):
            fig.add_trace(go.Scatter(x=e.Episode, y=e.Total, mode="markers", name=kind,
                                     marker=dict(color=color, symbol=sym, size=9)), 1, 2)

for col, name, color in [("W_Sens_Mot", "сенсор→мотор", "gold"), ("W_Sens_Hid", "сенсор→скрытые", "orange"),
                         ("W_Hid_Hid", "скрытые→скрытые", "violet"), ("W_Hid_Mot", "скрытые→мотор", "tomato")]:
    if col in ep and ep[col].abs().sum() > 0:
        fig.add_trace(go.Scatter(x=ep.Episode, y=ep[col], name=name, line=dict(color=color)), 2, 1)
for col, name, color in [("Rate_Sens", "сенсоры", "cyan"), ("Rate_Hid", "скрытые", "dodgerblue"), ("Rate_Mot", "моторы", "tomato")]:
    fig.add_trace(go.Scatter(x=ep.Episode, y=ep[col], name="акт. " + name, line=dict(color=color)), 2, 2)

fig.add_trace(go.Scatter(x=ep.Episode, y=ep.Miss_Distance.rolling(20, min_periods=1).mean(), name="промах",
                         line=dict(color="gold")), 3, 1)
fig.add_trace(go.Scatter(x=ep.Episode, y=ep.Synapses, name="синапсы", line=dict(color="mediumpurple")), 3, 2)

fig.update_yaxes(range=[0, 105], row=1, col=1)
fig.update_xaxes(title_text="Эпизод")
fig.update_layout(template="plotly_dark", height=900, title="Процесс обучения 3D-биочипа в Pong",
                  margin=dict(l=20, r=20, t=70, b=20))
fig.show()

if len(ev) and "Event" in ev:
    print(ev.Event.value_counts().to_string())
else:
    print("Структурных событий нет (нейрогенез/апоптоз выключены или не срабатывали).")
if os.path.exists("oi_runs.jsonl"):
    print("\nЖурнал запусков (последние 5):")
    for line in open("oi_runs.jsonl").read().strip().split("\n")[-5:]:
        r = json.loads(line)
        print(f"  {r['timestamp']} | {r['name']} | эпизодов {r['episodes_total']} | цель {r['reached']} | "
              f"последняя оценка {r['last_eval_pct']}% (случайная {r['baseline_random_pct']}%)")

# Панель среды: нейромедиаторы, множитель пластичности (дофамин), температура
if "Temp_C" in ep:
    f2 = make_subplots(rows=1, cols=3, subplot_titles=("Нейромедиаторы (отн. ед.)",
                                                       "Множитель скорости обучения (дофамин)", "Температура, °C"))
    for col, name, color in [("Glu", "глутамат", "orange"), ("GABA", "ГАМК", "deepskyblue"), ("DA_Level", "дофамин", "violet")]:
        f2.add_trace(go.Scatter(x=ep.Episode, y=ep[col], name=name, line=dict(color=color)), 1, 1)
    f2.add_trace(go.Scatter(x=ep.Episode, y=ep.Plasticity_Gain, name="lr x", line=dict(color="limegreen")), 1, 2)
    f2.add_trace(go.Scatter(x=ep.Episode, y=ep.Temp_C, name="T", line=dict(color="tomato")), 1, 3)
    f2.update_xaxes(title_text="Эпизод")
    f2.update_layout(template="plotly_dark", height=320, margin=dict(l=20, r=20, t=60, b=20))
    f2.show()

In [ ]:
# Ячейка 7: выгрузка СВОДОК в Google Таблицу. Покадровые данные (oi_frames.csv) не отправляются: их много.
import os, json
import numpy as np
import pandas as pd
from google.colab import auth
import gspread
from google.auth import default

SHEET_NAME = "OI_Pong_Game_Telemetry"
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)
try:
    sh = gc.open(SHEET_NAME)
except gspread.SpreadsheetNotFound:
    sh = gc.create(SHEET_NAME)
print("Таблица:", sh.url)

def push(title, df, chunk=4000):
    if df is None or len(df) == 0:
        return
    try:
        ws = sh.worksheet(title); ws.clear()
        ws.resize(rows=max(len(df) + 1, 2), cols=len(df.columns))
    except gspread.WorksheetNotFound:
        ws = sh.add_worksheet(title=title, rows=max(len(df) + 1, 2), cols=len(df.columns))
    df = df.copy()
    df = df.astype(object).where(df.notna(), "")
    ws.update(values=[df.columns.tolist()], range_name="A1")
    for start in range(0, len(df), chunk):
        part = df.iloc[start:start + chunk]
        part = [[(x.item() if hasattr(x, "item") else x) for x in row] for row in part.values.tolist()]
        ws.update(values=part, range_name=f"A{start + 2}")
    print(f"  {title}: {len(df)} строк")

for title, path in [("Eval", "oi_eval.csv"), ("Episodes", "oi_episodes.csv"), ("Structure_Events", "oi_events.csv"),
                    ("Neurons", "oi_neurons.csv"), ("Synapses", "oi_synapses.csv")]:
    if os.path.exists(path) and os.path.getsize(path) > 0:
        try:
            push(title, pd.read_csv(path))
        except pd.errors.EmptyDataError:
            print(f"  {title}: файл пуст, пропущен")

if os.path.exists("oi_runs.jsonl"):
    rows = [json.loads(l) for l in open("oi_runs.jsonl").read().strip().split("\n") if l.strip()]
    for r in rows:
        r["cfg"] = json.dumps(r.get("cfg", {}), ensure_ascii=False)
    push("Run_Log", pd.DataFrame(rows))
print("Готово.")

In [ ]:
# Ячейка 8: химия и температура. Эксперименты над ОБУЧЕННОЙ сетью (oi_state.pkl), без дообучения.
import numpy as np
from oi_engine import *

game = load_game("oi_state.pkl")
env = game.env
print("Среда сейчас:", env.snapshot())

# 1) Температура: Q10 меняет кинетику каналов -> число спайков -> скорость ракетки. Оценка без обучения.
print("\nТемпература (win rate обученной сети, 100 эпизодов без обучения):")
T0 = env.T
for T in (25.0, 30.0, 34.0, 37.0, 40.0, 42.0):
    env.set_temperature(T)
    print(f"  T = {T:4.1f} °C | кинетика x{env.kin_factor:5.2f} | win = {100*game.evaluate(100)['win']:5.1f}%")
env.set_temperature(T0)

# 2) Нейромедиаторы: глутамат усиливает возбуждающие синапсы, ГАМК - тормозные.
print("\nГлутамат (фоновая концентрация, отн. ед.):")
for glu in (0.3, 0.6, 1.0, 1.5, 2.5):
    env.set_tonic("glu", glu); env.level["glu"] = glu
    print(f"  Glu = {glu:3.1f} | усиление возбуждающих синапсов x{env.glu_gain:4.2f} | win = {100*game.evaluate(100)['win']:5.1f}%")
env.set_tonic("glu", 1.0); env.level["glu"] = 1.0

# 3) Дофамин: управляет скоростью пластичности (lr x gain). 0 = обучение выключено.
print("\nДофамин -> множитель скорости обучения (функция Хилла, 1.0 = норма):")
for da in (0.0, 0.1, 0.3, 1.0, 2.0):
    env.set_tonic("da", da); env.level["da"] = da
    print(f"  DA = {da:3.1f} -> множитель lr = {env.plasticity_gain:4.2f}")
env.set_tonic("da", 1.0); env.level["da"] = 1.0
print("\nДля влияния дофамина на САМО обучение: dopamine_scan() (ячейка 3, RUN_DOPAMINE_SCAN).")